### **Installations and Imports**

In [1]:
# ============================================================
# Cell 1 — Installations and Imports for ntkmirror experiment
# ============================================================

import importlib.util
import subprocess
import sys
from importlib.metadata import version as pkg_version, PackageNotFoundError

def is_installed(import_name):
    return importlib.util.find_spec(import_name) is not None

required = {
    "transformers": "transformers",
    "trl": "trl",
    "accelerate": "accelerate",
    "bitsandbytes": "bitsandbytes",
    "datasets": "datasets",
    "sacrebleu": "sacrebleu",
    "sentencepiece": "sentencepiece",
    "packaging": "packaging",
    "tqdm": "tqdm",
    "pandas": "pandas",
    "openpyxl": "openpyxl",
}

missing = [
    pip_name
    for pip_name, import_name in required.items()
    if not is_installed(import_name)
]

print("Missing packages:", missing)

if missing:
    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        *missing,
    ]
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)

# Install ntkmirror directly from the repo.
if not is_installed("ntkmirror"):
    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        "git+https://github.com/leochlon/ntkmirror.git",
    ]
    print("Installing ntkmirror:")
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)
else:
    print("ntkmirror is already installed.")

from packaging.version import parse as parse_version

def installed_version(package_name):
    try:
        return pkg_version(package_name)
    except PackageNotFoundError:
        return None

transformers_version = installed_version("transformers")
print("Current transformers:", transformers_version)

if transformers_version is None or parse_version(transformers_version) < parse_version("4.51.0"):
    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "transformers>=4.51.0",
    ]
    print("Upgrading transformers for Qwen3 support:")
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)

print("Installation finished.")

Missing packages: ['trl', 'bitsandbytes', 'sacrebleu']
Running: /usr/bin/python3 -m pip install -q --no-cache-dir trl bitsandbytes sacrebleu
Installing ntkmirror:
Running: /usr/bin/python3 -m pip install -q --no-cache-dir git+https://github.com/leochlon/ntkmirror.git
Current transformers: 5.10.2
Installation finished.


In [2]:
# ============================================================
# Cell 1B — Verify environment
# ============================================================

import torch
import datasets
import transformers
import peft
import accelerate
import trl
import bitsandbytes
import sacrebleu

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("WARNING: CUDA is not available.")
    print("Do not start Qwen/Unsloth fine-tuning on CPU.")
    print("Go to Runtime → Change runtime type → GPU.")

print("datasets:", datasets.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)
print("trl:", trl.__version__)
print("sacrebleu:", sacrebleu.__version__)

print("Environment check finished.")

Torch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8
datasets: 5.0.0
transformers: 5.10.2
peft: 0.19.1
accelerate: 1.13.0
trl: 1.6.0
sacrebleu: 2.6.0
Environment check finished.


### **Paths and Configurations**

In [3]:
# ============================================================
# Cell 2 — Mount Google Drive and define paths
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/alexandria_qwen35_sft")
DATA_DIR    = PROJECT_DIR / "prepared_data"
RUNS_DIR    = PROJECT_DIR / "runs"
ADAPTER_DIR = PROJECT_DIR / "final_adapters"
PRED_DIR    = PROJECT_DIR / "predictions"

for p in [PROJECT_DIR, DATA_DIR, RUNS_DIR, ADAPTER_DIR, PRED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR:", DATA_DIR)
print("RUNS_DIR:", RUNS_DIR)
print("ADAPTER_DIR:", ADAPTER_DIR)
print("PRED_DIR:", PRED_DIR)

Mounted at /content/drive
PROJECT_DIR: /content/drive/MyDrive/alexandria_qwen35_sft
DATA_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/prepared_data
RUNS_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/runs
ADAPTER_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/final_adapters
PRED_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/predictions


In [4]:
# ============================================================
# Cell 3 — Experiment configuration
# NileChat-3B + ntkmirror controller
# EG-only, context3, complete-2shot, same checkpoint/metric constraints
# ============================================================

import torch
import random
import numpy as np
from pathlib import Path

SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

# NileChat is a 3B dialect-aware model, suitable for <=5B constraint.
# IMPORTANT: You must accept the HF model conditions first if access is gated.
MODEL_NAME = "UBC-NLP/NileChat-3B"
LOAD_IN_4BIT = True

# ------------------------------------------------------------
# Country/config selection
# ------------------------------------------------------------

SELECTED_CONFIGS_MODE = "EG_ONLY"
MANUAL_CONFIGS = ["EG"]

# ------------------------------------------------------------
# Translation setup
# ------------------------------------------------------------

MAX_CONTEXT_TURNS = 3
USE_PREVIOUS_ENGLISH_CONTEXT = True
USE_METADATA = True

# ------------------------------------------------------------
# Few-shot setup
# ------------------------------------------------------------

USE_FEW_SHOTS = True
N_FEW_SHOTS = 2
MAX_FEW_SHOT_EXAMPLE_CHARS = 450

# NileChat model card uses sequence length 4096 in training, but keep 2048
# for Colab stability and consistency with your current notebook.
MAX_SEQ_LENGTH = 2048

# ------------------------------------------------------------
# ntkmirror controller settings
# Same strong controlled setup as the improved Qwen run
# ------------------------------------------------------------

NTK_GATES = 10000
NTK_LAYERS = "all"
NTK_MAX_LOG_GATE = 0.10
NTK_HOOK_SITE = "layer_output"

NTK_SCORE_EXAMPLES = 256
NTK_SCORE_BATCHES = 256

NTK_L2 = 1e-5

# ------------------------------------------------------------
# Training schedule
# ------------------------------------------------------------

NUM_EPOCHS = 10

PER_DEVICE_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8

LEARNING_RATE = 1e-3
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01

SAVE_STEPS = 100
EVAL_STEPS = 100
LOGGING_STEPS = 10

SAVE_TOTAL_LIMIT = 50

# Keep full eval NLL for best checkpoint selection.
# If too slow, temporarily set to 200, but for final run keep None.
EVAL_NLL_LIMIT = None

# ------------------------------------------------------------
# Experiment naming
# New name = safe paths, no collision with Qwen runs
# ------------------------------------------------------------

EXPERIMENT_NAME = (
    f"nilechat3b_alexandria_{SELECTED_CONFIGS_MODE.lower()}_"
    f"context{MAX_CONTEXT_TURNS}_complete2shot_ntkmirror_"
    f"g{NTK_GATES}_score{NTK_SCORE_EXAMPLES}_"
    f"mlgate010_lr1e3_all_10epochs"
)

OUTPUT_DIR = RUNS_DIR / EXPERIMENT_NAME
CONTROLLER_DIR = ADAPTER_DIR / EXPERIMENT_NAME

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CONTROLLER_DIR.mkdir(parents=True, exist_ok=True)

print("Model:", MODEL_NAME)
print("Load in 4-bit:", LOAD_IN_4BIT)
print("Experiment:", EXPERIMENT_NAME)

print("\nntkmirror config:")
print("  gates:", NTK_GATES)
print("  layers:", NTK_LAYERS)
print("  max_log_gate:", NTK_MAX_LOG_GATE)
print("  hook_site:", NTK_HOOK_SITE)
print("  score examples:", NTK_SCORE_EXAMPLES)
print("  score batches:", NTK_SCORE_BATCHES)
print("  l2:", NTK_L2)

print("\nPrompt/data config:")
print("  EG only:", SELECTED_CONFIGS_MODE)
print("  context turns:", MAX_CONTEXT_TURNS)
print("  Few-shot enabled:", USE_FEW_SHOTS)
print("  Few-shot examples:", N_FEW_SHOTS)
print("  Max few-shot example chars:", MAX_FEW_SHOT_EXAMPLE_CHARS)
print("  Max seq length:", MAX_SEQ_LENGTH)

print("\nTraining config:")
print("  Epochs:", NUM_EPOCHS)
print("  Batch size:", PER_DEVICE_BATCH_SIZE)
print("  Gradient accumulation:", GRAD_ACCUM_STEPS)
print("  Learning rate:", LEARNING_RATE)
print("  Save steps:", SAVE_STEPS)
print("  Eval steps:", EVAL_STEPS)
print("  Save total limit:", SAVE_TOTAL_LIMIT)

print("\nPaths:")
print("  Output dir:", OUTPUT_DIR)
print("  Controller dir:", CONTROLLER_DIR)

print("\nExisting checkpoints in this experiment:")
print(sorted([p.name for p in OUTPUT_DIR.glob("checkpoint-*")]))

Model: UBC-NLP/NileChat-3B
Load in 4-bit: True
Experiment: nilechat3b_alexandria_eg_only_context3_complete2shot_ntkmirror_g10000_score256_mlgate010_lr1e3_all_10epochs

ntkmirror config:
  gates: 10000
  layers: all
  max_log_gate: 0.1
  hook_site: layer_output
  score examples: 256
  score batches: 256
  l2: 1e-05

Prompt/data config:
  EG only: EG_ONLY
  context turns: 3
  Few-shot enabled: True
  Few-shot examples: 2
  Max few-shot example chars: 450
  Max seq length: 2048

Training config:
  Epochs: 10
  Batch size: 1
  Gradient accumulation: 8
  Learning rate: 0.001
  Save steps: 100
  Eval steps: 100
  Save total limit: 50

Paths:
  Output dir: /content/drive/MyDrive/alexandria_qwen35_sft/runs/nilechat3b_alexandria_eg_only_context3_complete2shot_ntkmirror_g10000_score256_mlgate010_lr1e3_all_10epochs
  Controller dir: /content/drive/MyDrive/alexandria_qwen35_sft/final_adapters/nilechat3b_alexandria_eg_only_context3_complete2shot_ntkmirror_g10000_score256_mlgate010_lr1e3_all_10epoch

In [5]:
print("EXPERIMENT_NAME:", EXPERIMENT_NAME)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("CONTROLLER_DIR:", CONTROLLER_DIR)

print("\nExisting checkpoints in this new run:")
print(sorted([p.name for p in OUTPUT_DIR.glob("checkpoint-*")]))

old_name = "qwen3_4b_alexandria_eg_only_context3_complete2shot_ntkmirror_g5000_10epochs"
print("\nOld run name appears in new OUTPUT_DIR?", old_name in str(OUTPUT_DIR))
print("Old run name appears in new CONTROLLER_DIR?", old_name in str(CONTROLLER_DIR))

EXPERIMENT_NAME: nilechat3b_alexandria_eg_only_context3_complete2shot_ntkmirror_g10000_score256_mlgate010_lr1e3_all_10epochs
OUTPUT_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/runs/nilechat3b_alexandria_eg_only_context3_complete2shot_ntkmirror_g10000_score256_mlgate010_lr1e3_all_10epochs
CONTROLLER_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/final_adapters/nilechat3b_alexandria_eg_only_context3_complete2shot_ntkmirror_g10000_score256_mlgate010_lr1e3_all_10epochs

Existing checkpoints in this new run:
[]

Old run name appears in new OUTPUT_DIR? False
Old run name appears in new CONTROLLER_DIR? False


### **Dataset Preparation**

In [6]:
# ============================================================
# Cell 4 — List Alexandria configs and load selected configs
# ============================================================

from datasets import load_dataset, get_dataset_config_names
import pandas as pd

DATASET_NAME = "UBC-NLP/alexandria"

available_configs = get_dataset_config_names(DATASET_NAME)
print("Available Alexandria configs:")
print(available_configs)

if SELECTED_CONFIGS_MODE == "EG_ONLY":
    selected_configs = ["EG"] if "EG" in available_configs else [available_configs[0]]

elif SELECTED_CONFIGS_MODE == "ALL":
    selected_configs = available_configs

elif SELECTED_CONFIGS_MODE == "MANUAL":
    selected_configs = MANUAL_CONFIGS
    missing = [c for c in selected_configs if c not in available_configs]
    if missing:
        raise ValueError(f"These configs are not available: {missing}")

else:
    raise ValueError("SELECTED_CONFIGS_MODE must be EG_ONLY, ALL, or MANUAL.")

print("\nSelected configs:")
print(selected_configs)

loaded = {}

for cfg in selected_configs:
    print(f"\nLoading config: {cfg}")
    ds_train = load_dataset(DATASET_NAME, name=cfg, split="train")
    ds_test  = load_dataset(DATASET_NAME, name=cfg, split="test")

    loaded[cfg] = {
        "train": ds_train,
        "test": ds_test,
    }

    print("Train:", ds_train)
    print("Test:", ds_test)
    print("Example keys:", ds_train[0].keys())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/24.5k [00:00<?, ?B/s]

Available Alexandria configs:
['EG', 'JO', 'LB', 'LY', 'MA', 'MR', 'OM', 'PS', 'SA', 'SD', 'SY', 'TN', 'YE']

Selected configs:
['EG']

Loading config: EG


EG/train-00000-of-00001.parquet:   0%|          | 0.00/496k [00:00<?, ?B/s]

EG/test-00000-of-00001.parquet:   0%|          | 0.00/196k [00:00<?, ?B/s]

EG/dev-00000-of-00001.parquet:   0%|          | 0.00/183k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/982 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/366 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/352 [00:00<?, ? examples/s]

Train: Dataset({
    features: ['conv_id', 'country', 'domain', 'dialect', 'participants', 'english_conversation', 'dialectal_conversation', 'translator_id', 'reviewer_id'],
    num_rows: 982
})
Test: Dataset({
    features: ['conv_id', 'country', 'domain', 'dialect', 'participants', 'english_conversation', 'dialectal_conversation', 'translator_id', 'reviewer_id'],
    num_rows: 366
})
Example keys: dict_keys(['conv_id', 'country', 'domain', 'dialect', 'participants', 'english_conversation', 'dialectal_conversation', 'translator_id', 'reviewer_id'])


In [7]:
# ============================================================
# Cell 5 — Inspect one raw example
# ============================================================

sample_cfg = selected_configs[0]
sample_row = loaded[sample_cfg]["train"][0]

print("Config:", sample_cfg)
print("Keys:", sample_row.keys())

print("\nEnglish conversation:")
print(sample_row["english_conversation"])

print("\nDialectal conversation:")
print(sample_row["dialectal_conversation"])

print("\nFull row:")
sample_row

Config: EG
Keys: dict_keys(['conv_id', 'country', 'domain', 'dialect', 'participants', 'english_conversation', 'dialectal_conversation', 'translator_id', 'reviewer_id'])

English conversation:
[{'direction': 'male -> female', 'speaker': 'Wholesale Buyer', 'text': "Good morning. I'm looking to source 10 tons of premium artichokes for export. People say the best quality in Obour comes from your section, is that right?", 'turn_order': 1}, {'direction': 'female -> male', 'speaker': 'Wholesale Seller', 'text': "Good morning to you. You heard correctly. My artichokes are the best you'll find. They are top-grade, perfect for export. Let me show you a sample.", 'turn_order': 2}, {'direction': 'male -> female', 'speaker': 'Wholesale Buyer', 'text': 'Excellent. Yes, please show me. I need them to be a specific size and completely free of blemishes.', 'turn_order': 3}, {'direction': 'female -> male', 'speaker': 'Wholesale Seller', 'text': "Don't you worry. You will be very satisfied. My reputatio

{'conv_id': 'B7-1-0-120',
 'country': 'EG',
 'domain': 'Agriculture and farming',
 'dialect': 'Egyptian Arabic (Cairene) Dialect',
 'participants': 'Wholesale Buyer, Wholesale Seller',
 'english_conversation': [{'direction': 'male -> female',
   'speaker': 'Wholesale Buyer',
   'text': "Good morning. I'm looking to source 10 tons of premium artichokes for export. People say the best quality in Obour comes from your section, is that right?",
   'turn_order': 1},
  {'direction': 'female -> male',
   'speaker': 'Wholesale Seller',
   'text': "Good morning to you. You heard correctly. My artichokes are the best you'll find. They are top-grade, perfect for export. Let me show you a sample.",
   'turn_order': 2},
  {'direction': 'male -> female',
   'speaker': 'Wholesale Buyer',
   'text': 'Excellent. Yes, please show me. I need them to be a specific size and completely free of blemishes.',
   'turn_order': 3},
  {'direction': 'female -> male',
   'speaker': 'Wholesale Seller',
   'text': "D

#### Helper functions for robust extraction

In [8]:
# ============================================================
# Cell 6 — Helper functions for robust extraction
# ============================================================

def safe_get(row, keys, default=""):
    for k in keys:
        if isinstance(row, dict) and k in row and row[k] is not None:
            return row[k]
    return default

def turn_text(turn):
    if isinstance(turn, dict):
        for k in ["text", "sentence", "utterance", "content", "value"]:
            if k in turn and turn[k] is not None:
                return str(turn[k]).strip()
        return str(turn).strip()
    return str(turn).strip()

def turn_field(turn, keys, default=""):
    if isinstance(turn, dict):
        for k in keys:
            if k in turn and turn[k] is not None:
                return str(turn[k]).strip()
    return default

def normalize_list(x):
    if x is None:
        return []
    if isinstance(x, list):
        return x
    return list(x)

def truncate_text(text, max_chars=1200):
    text = str(text)
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rstrip() + " ..."

#### Flatten Alexandria conversations into SFT examples and Save

In [9]:
# ============================================================
# Cell 7 — Flatten Alexandria conversations
# ============================================================

def flatten_alexandria_split(ds, cfg_name, split_name, max_context_turns=3):
    records = []

    for conv_idx, row in enumerate(ds):
        english_conv = normalize_list(row["english_conversation"])
        dialect_conv = normalize_list(row["dialectal_conversation"])

        n = min(len(english_conv), len(dialect_conv))

        country = safe_get(row, ["country", "country_code"], cfg_name)
        dialect = safe_get(row, ["dialect", "dialect_label", "subdialect", "city", "variety"], "")
        domain = safe_get(row, ["domain", "topic"], "")
        persona = safe_get(row, ["persona", "roles", "speaker_roles"], "")
        conv_id = safe_get(row, ["conversation_id", "id", "dialogue_id"], f"{cfg_name}_{split_name}_{conv_idx}")

        for i in range(n):
            en_turn = english_conv[i]
            ar_turn = dialect_conv[i]

            source_text = turn_text(en_turn)
            target_text = turn_text(ar_turn)

            if not source_text or not target_text:
                continue

            prev_start = max(0, i - max_context_turns)
            prev_en_turns = english_conv[prev_start:i]

            previous_context = []
            for j, t in enumerate(prev_en_turns, start=prev_start):
                previous_context.append({
                    "turn_id": j,
                    "speaker": turn_field(t, ["speaker", "role", "speaker_role"], ""),
                    "direction": turn_field(t, ["direction", "gender_direction", "speaker_addressee_gender"], ""),
                    "text": turn_text(t),
                })

            records.append({
                "source_id": f"{cfg_name}_{split_name}_{conv_id}_{i}",
                "config": cfg_name,
                "split": split_name,
                "conversation_id": conv_id,
                "turn_id": i,

                "country": country,
                "dialect": dialect,
                "domain": domain,
                "persona": persona,

                "speaker": turn_field(en_turn, ["speaker", "role", "speaker_role"], ""),
                "gender_direction": turn_field(en_turn, ["direction", "gender_direction", "speaker_addressee_gender"], ""),

                "previous_english_turns": previous_context,
                "source_text": source_text,
                "target_arabic": target_text,
            })

    return records

train_records = []
eval_records = []

for cfg in selected_configs:
    train_records.extend(
        flatten_alexandria_split(
            loaded[cfg]["train"],
            cfg_name=cfg,
            split_name="train",
            max_context_turns=MAX_CONTEXT_TURNS,
        )
    )

    eval_records.extend(
        flatten_alexandria_split(
            loaded[cfg]["test"],
            cfg_name=cfg,
            split_name="test",
            max_context_turns=MAX_CONTEXT_TURNS,
        )
    )

train_df = pd.DataFrame(train_records)
eval_df = pd.DataFrame(eval_records)

print("Train shape:", train_df.shape)
print("Eval shape:", eval_df.shape)

print("\nTrain configs:")
print(train_df["config"].value_counts())

print("\nEval configs:")
print(eval_df["config"].value_counts())

display(train_df.head())

Train shape: (3108, 14)
Eval shape: (1118, 14)

Train configs:
config
EG    3108
Name: count, dtype: int64

Eval configs:
config
EG    1118
Name: count, dtype: int64


,source_id,config,split,conversation_id,turn_id,country,dialect,domain,persona,speaker,gender_direction,previous_english_turns,source_text,target_arabic
0,EG_train_EG_train_0_0,EG,train,EG_train_0,0,EG,Egyptian Arabic (Cairene) Dialect,Agriculture and farming,,Wholesale Buyer,male -> female,[],Good morning. I'm looking to source 10 tons of...,صباح الخير، عايز عشرة طن من الخرشوف الكويس للت...
1,EG_train_EG_train_0_1,EG,train,EG_train_0,1,EG,Egyptian Arabic (Cairene) Dialect,Agriculture and farming,,Wholesale Seller,female -> male,"[{'turn_id': 0, 'speaker': 'Wholesale Buyer', ...",Good morning to you. You heard correctly. My a...,صباح النور،سمعك مظبوط،الخرشوف بتاعي من أحسن ال...
2,EG_train_EG_train_0_2,EG,train,EG_train_0,2,EG,Egyptian Arabic (Cairene) Dialect,Agriculture and farming,,Wholesale Buyer,male -> female,"[{'turn_id': 0, 'speaker': 'Wholesale Buyer', ...","Excellent. Yes, please show me. I need them to...",ممتاز، لو سمحتي وريني، عايزه بمقاس واحد ومافيه...
3,EG_train_EG_train_0_3,EG,train,EG_train_0,3,EG,Egyptian Arabic (Cairene) Dialect,Agriculture and farming,,Wholesale Seller,female -> male,"[{'turn_id': 0, 'speaker': 'Wholesale Buyer', ...",Don't you worry. You will be very satisfied. M...,متخافش، هتنبسط جدا، سمعتي جاية من الحاجة الكويسة.
4,EG_train_EG_train_1_0,EG,train,EG_train_1,0,EG,Egyptian Arabic (Cairene) Dialect,Agriculture and farming,,Farmer,female -> female,[],I usually use the regular granular fertilizer....,أنا عادة بستخدم السماد العادي الحبيبات. ايه فا...


In [10]:
# ============================================================
# Cell 8 — Save prepared flattened data
# ============================================================

train_jsonl = DATA_DIR / f"alexandria_train_{SELECTED_CONFIGS_MODE.lower()}_context{MAX_CONTEXT_TURNS}.jsonl"
eval_jsonl  = DATA_DIR / f"alexandria_eval_{SELECTED_CONFIGS_MODE.lower()}_context{MAX_CONTEXT_TURNS}.jsonl"

train_df.to_json(train_jsonl, orient="records", lines=True, force_ascii=False)
eval_df.to_json(eval_jsonl, orient="records", lines=True, force_ascii=False)

print("Saved train:", train_jsonl)
print("Saved eval:", eval_jsonl)

Saved train: /content/drive/MyDrive/alexandria_qwen35_sft/prepared_data/alexandria_train_eg_only_context3.jsonl
Saved eval: /content/drive/MyDrive/alexandria_qwen35_sft/prepared_data/alexandria_eval_eg_only_context3.jsonl


### **Build prompt and chat messages**

In [11]:
# ============================================================
# Cell 9 — Build prompt and chat messages
# Complete 2-shot examples from training data only
# ============================================================

from datasets import Dataset
import hashlib
import pandas as pd

SYSTEM_PROMPT = (
    "You are a professional machine translation system. "
    "Translate the current English dialogue turn into natural dialectal Arabic. "
    "Use the provided training examples only as style and dialect guidance. "
    "Return only the translation, without explanation."
)

def build_context(previous_turns):
    if not USE_PREVIOUS_ENGLISH_CONTEXT or not previous_turns:
        return "No previous context."

    lines = []
    for i, t in enumerate(previous_turns, start=1):
        speaker = t.get("speaker", "")
        text = t.get("text", "")

        if speaker:
            lines.append(f"{i}. {speaker}: {text}")
        else:
            lines.append(f"{i}. {text}")

    return "\n".join(lines)

def build_metadata_block(row):
    if not USE_METADATA:
        return "No metadata."

    fields = [
        ("Country/config", row.get("config", "")),
        ("Target dialect", row.get("dialect", "")),
        ("Domain", row.get("domain", "")),
        ("Persona/Roles", row.get("persona", "")),
        ("Current speaker", row.get("speaker", "")),
        ("Speaker-to-addressee gender direction", row.get("gender_direction", "")),
    ]

    lines = []
    for k, v in fields:
        v = str(v).strip()
        if v:
            lines.append(f"{k}: {v}")

    return "\n".join(lines) if lines else "No metadata."

def deterministic_seed_from_id(source_id, base_seed=SEED):
    raw = f"{source_id}_{base_seed}".encode("utf-8")
    return int(hashlib.md5(raw).hexdigest()[:8], 16)

def select_two_shots_from_train(row, train_pool, n=N_FEW_SHOTS):
    """
    Select n COMPLETE examples from training data only.

    The examples are NOT truncated.

    To keep MAX_SEQ_LENGTH reasonable, we prefer naturally short examples:
        source_text length + target_arabic length <= MAX_FEW_SHOT_EXAMPLE_CHARS

    Priority:
    1. same config + same domain + short
    2. same config + short
    3. any short
    4. same config + same domain
    5. same config
    6. any training example

    For train rows, exclude the same source_id to avoid using itself as a shot.
    """
    if not USE_FEW_SHOTS or n <= 0:
        return []

    row_source_id = str(row.get("source_id", ""))
    row_config = str(row.get("config", ""))
    row_domain = str(row.get("domain", ""))

    pool = train_pool.copy()
    pool["source_id"] = pool["source_id"].astype(str)

    # Avoid using the same train row as its own few-shot example.
    pool = pool[pool["source_id"] != row_source_id].copy()

    if len(pool) == 0:
        return []

    pool["fewshot_total_chars"] = (
        pool["source_text"].astype(str).str.len()
        + pool["target_arabic"].astype(str).str.len()
    )

    short_pool = pool[pool["fewshot_total_chars"] <= MAX_FEW_SHOT_EXAMPLE_CHARS].copy()

    same_config_domain_short = short_pool[
        (short_pool["config"].astype(str) == row_config)
        & (short_pool["domain"].astype(str) == row_domain)
    ]

    same_config_short = short_pool[
        short_pool["config"].astype(str) == row_config
    ]

    same_config_domain = pool[
        (pool["config"].astype(str) == row_config)
        & (pool["domain"].astype(str) == row_domain)
    ]

    same_config = pool[
        pool["config"].astype(str) == row_config
    ]

    candidate_pools = [
        same_config_domain_short,
        same_config_short,
        short_pool,
        same_config_domain,
        same_config,
        pool,
    ]

    candidates = None
    for candidate_pool in candidate_pools:
        if len(candidate_pool) >= n:
            candidates = candidate_pool
            break

    if candidates is None:
        candidates = pool

    sample_n = min(n, len(candidates))
    seed = deterministic_seed_from_id(row_source_id)

    shots = candidates.sample(n=sample_n, random_state=seed)

    keep_cols = [
        "source_id",
        "config",
        "dialect",
        "domain",
        "source_text",
        "target_arabic",
    ]

    return shots[keep_cols].to_dict("records")

def build_few_shot_block(few_shot_examples):
    if not USE_FEW_SHOTS or not few_shot_examples:
        return "No examples available."

    blocks = []

    for i, ex in enumerate(few_shot_examples, start=1):
        ex_config = str(ex.get("config", "")).strip()
        ex_dialect = str(ex.get("dialect", "")).strip()
        ex_domain = str(ex.get("domain", "")).strip()

        meta_parts = []
        if ex_config:
            meta_parts.append(f"config={ex_config}")
        if ex_dialect:
            meta_parts.append(f"dialect={ex_dialect}")
        if ex_domain:
            meta_parts.append(f"domain={ex_domain}")

        meta_line = ", ".join(meta_parts) if meta_parts else "no metadata"

        # COMPLETE examples.
        # No truncation here.
        ex_source = str(ex.get("source_text", "")).strip()
        ex_target = str(ex.get("target_arabic", "")).strip()

        blocks.append(
            f"""Example {i} ({meta_line})
English:
{ex_source}

Arabic:
{ex_target}"""
        )

    return "\n\n".join(blocks)

def make_user_prompt(row):
    context = build_context(row["previous_english_turns"])
    metadata = build_metadata_block(row)
    few_shots = build_few_shot_block(row.get("few_shot_examples", []))

    return f"""Task:
Translate the current English dialogue turn into the target dialectal Arabic variety.

Few-shot training examples:
{few_shots}

Metadata:
{metadata}

Previous English dialogue context:
{context}

Current English turn:
{row["source_text"]}

Rules:
- Preserve the meaning exactly.
- Use the target local dialect, not Modern Standard Arabic unless it is natural in context.
- Follow the dialect/style pattern shown in the few-shot examples when relevant.
- Do not copy the few-shot examples.
- Preserve names, numbers, named entities, and technical terms when appropriate.
- Keep the tone appropriate for the speaker and domain.
- Return only the Arabic translation."""

def row_to_messages(row):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_prompt(row)},
        {"role": "assistant", "content": row["target_arabic"]},
    ]

# ------------------------------------------------------------
# Build few-shot pool from training data only
# ------------------------------------------------------------

few_shot_pool_cols = [
    "source_id",
    "config",
    "dialect",
    "domain",
    "source_text",
    "target_arabic",
]

missing_few_shot_cols = [
    c for c in few_shot_pool_cols
    if c not in train_df.columns
]

if missing_few_shot_cols:
    raise ValueError(f"Missing columns in train_df for few-shot selection: {missing_few_shot_cols}")

train_few_shot_pool = train_df[few_shot_pool_cols].copy()

train_df["few_shot_examples"] = train_df.apply(
    lambda row: select_two_shots_from_train(row, train_few_shot_pool, n=N_FEW_SHOTS),
    axis=1,
)

eval_df["few_shot_examples"] = eval_df.apply(
    lambda row: select_two_shots_from_train(row, train_few_shot_pool, n=N_FEW_SHOTS),
    axis=1,
)

train_df["messages"] = train_df.apply(row_to_messages, axis=1)
eval_df["messages"]  = eval_df.apply(row_to_messages, axis=1)

train_dataset = Dataset.from_pandas(
    train_df[
        [
            "messages",
            "source_id",
            "config",
            "domain",
            "dialect",
            "source_text",
            "target_arabic",
        ]
    ],
    preserve_index=False,
)

eval_dataset = Dataset.from_pandas(
    eval_df[
        [
            "messages",
            "source_id",
            "config",
            "domain",
            "dialect",
            "source_text",
            "target_arabic",
        ]
    ],
    preserve_index=False,
)

print(train_dataset)
print(eval_dataset)

print("\nExample few-shot source IDs for first train row:")
print([x["source_id"] for x in train_df.iloc[0]["few_shot_examples"]])

print("\nFew-shot example lengths for first train row:")
for i, ex in enumerate(train_df.iloc[0]["few_shot_examples"], start=1):
    total_chars = len(str(ex["source_text"])) + len(str(ex["target_arabic"]))
    print(f"Example {i}: source_id={ex['source_id']}, total_chars={total_chars}")

print("\nExample messages:")
train_dataset[0]["messages"]

Dataset({
    features: ['messages', 'source_id', 'config', 'domain', 'dialect', 'source_text', 'target_arabic'],
    num_rows: 3108
})
Dataset({
    features: ['messages', 'source_id', 'config', 'domain', 'dialect', 'source_text', 'target_arabic'],
    num_rows: 1118
})

Example few-shot source IDs for first train row:
['EG_train_EG_train_46_2', 'EG_train_EG_train_67_0']

Few-shot example lengths for first train row:
Example 1: source_id=EG_train_EG_train_46_2, total_chars=243
Example 2: source_id=EG_train_EG_train_67_0, total_chars=75

Example messages:


[{'role': 'system',
  'content': 'You are a professional machine translation system. Translate the current English dialogue turn into natural dialectal Arabic. Use the provided training examples only as style and dialect guidance. Return only the translation, without explanation.'},
 {'role': 'user',
  'content': "Task:\nTranslate the current English dialogue turn into the target dialectal Arabic variety.\n\nFew-shot training examples:\nExample 1 (config=EG, dialect=Egyptian Arabic (Cairene) Dialect, domain=Agriculture and farming)\nEnglish:\nBy monitoring, we'll know exactly when the pest levels are high enough to justify spraying. This saves you money and protects the environment.\n\nArabic:\nلما نراقب، هنعرف بالضبط امتى مستويات الحشرات عالية كفاية عشان نبرر الرش. ده بيوفر فلوس وبيحمي البيئة.\n\nExample 2 (config=EG, dialect=Egyptian Arabic (Cairene) Dialect, domain=Agriculture and farming)\nEnglish:\nSo, what will this cost me for the whole job?\n\nArabic:\nطيب قد ايه الشغل كله هيكل

### **Load NileChat-3B for ntkmirror**

In [14]:
# ============================================================
# Cell 9B — Hugging Face login for gated models
# Required for UBC-NLP/NileChat-3B
# ============================================================

import os
import getpass
from huggingface_hub import login, whoami

HF_MODEL_REQUIRES_AUTH = True
HF_TOKEN = None

# Preferred on Colab:
# Runtime sidebar -> Secrets -> add:
# Name: HF_TOKEN
# Value: your Hugging Face token
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

# Fallback: manually paste token without showing it
if HF_TOKEN is None or str(HF_TOKEN).strip() == "":
    print("No HF_TOKEN found in Colab secrets.")
    HF_TOKEN = getpass.getpass("Paste your Hugging Face token here: ").strip()

if HF_TOKEN is None or HF_TOKEN == "":
    raise RuntimeError("HF token is missing.")

login(token=HF_TOKEN, add_to_git_credential=False)

try:
    user_info = whoami(token=HF_TOKEN)
    print("Logged in to Hugging Face as:", user_info.get("name", "unknown"))
except Exception as e:
    raise RuntimeError(
        "Hugging Face login failed. Check your token."
    ) from e

print("\nIMPORTANT:")
print("If NileChat-3B is gated, you must also open the model page and accept access conditions once.")
print("Model:", MODEL_NAME)

No HF_TOKEN found in Colab secrets.
Paste your Hugging Face token here: ··········
Logged in to Hugging Face as: MohamedAbdallah98

IMPORTANT:
If NileChat-3B is gated, you must also open the model page and accept access conditions once.
Model: UBC-NLP/NileChat-3B


In [15]:
# ============================================================
# Cell 10 — Load NileChat-3B for ntkmirror
# Auth-aware loading for gated Hugging Face model
# No Unsloth / no PEFT / no LoRA
# ============================================================

import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# ------------------------------------------------------------
# Check HF token
# ------------------------------------------------------------

if "HF_TOKEN" not in globals() or HF_TOKEN is None or str(HF_TOKEN).strip() == "":
    raise RuntimeError(
        "HF_TOKEN is missing. Run Cell 9B first, then rerun Cell 10.\n"
        "Also make sure you accepted access to UBC-NLP/NileChat-3B on Hugging Face."
    )

HF_TOKEN = str(HF_TOKEN).strip()

# ------------------------------------------------------------
# Quantization config
# ------------------------------------------------------------

if LOAD_IN_4BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
else:
    quantization_config = None

# ------------------------------------------------------------
# Load tokenizer
# ------------------------------------------------------------

print("Loading tokenizer:", MODEL_NAME)

try:
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        trust_remote_code=True,
        token=HF_TOKEN,
    )
except Exception as e:
    raise RuntimeError(
        f"Failed to load tokenizer for {MODEL_NAME}.\n\n"
        "Most likely causes:\n"
        "1. You did not run Cell 9B successfully.\n"
        "2. Your HF token does not have access.\n"
        "3. You did not accept the gated model conditions on Hugging Face.\n\n"
        "Fix:\n"
        "- Open https://huggingface.co/UBC-NLP/NileChat-3B while logged in.\n"
        "- Accept/request access.\n"
        "- Rerun Cell 9B.\n"
        "- Rerun Cell 10."
    ) from e

if tokenizer.pad_token is None:
    if tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token
    else:
        tokenizer.add_special_tokens({"pad_token": "<|pad|>"})

# ------------------------------------------------------------
# Load model
# ------------------------------------------------------------

print("Loading model:", MODEL_NAME)

try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quantization_config,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True,
        token=HF_TOKEN,
    )
except Exception as e:
    raise RuntimeError(
        f"Failed to load model for {MODEL_NAME}.\n\n"
        "Most likely causes:\n"
        "1. Gated model access was not accepted yet.\n"
        "2. HF token is invalid or lacks read permission.\n"
        "3. Colab runtime does not have enough memory.\n\n"
        "Fix gated access first if the error is 401 Unauthorized."
    ) from e

model.eval()
model.config.use_cache = False

# ------------------------------------------------------------
# If pad token was newly added, resize embeddings
# Usually not needed if pad=eos
# ------------------------------------------------------------

try:
    if len(tokenizer) > model.get_input_embeddings().weight.shape[0]:
        print("Resizing token embeddings because tokenizer length > embedding matrix.")
        model.resize_token_embeddings(len(tokenizer))
except Exception as e:
    print("WARNING: Could not check/resize token embeddings:", repr(e))

# ------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------

print("\nLoaded successfully.")
print("Model:", MODEL_NAME)
print("load_in_4bit:", LOAD_IN_4BIT)
print("model device:", next(model.parameters()).device)
print("model dtype:", next(model.parameters()).dtype)
print("tokenizer type:", type(tokenizer))
print("vocab size:", len(tokenizer))
print("pad token:", tokenizer.pad_token)
print("pad token id:", tokenizer.pad_token_id)
print("eos token:", tokenizer.eos_token)
print("eos token id:", tokenizer.eos_token_id)
print("has chat_template:", tokenizer.chat_template is not None)

if tokenizer.chat_template is None:
    print("\nWARNING: tokenizer has no chat_template. Cell 11 will fallback to manual prompt format.")
else:
    print("\nNileChat chat template is available and will be used in Cell 11.")

Loading tokenizer: UBC-NLP/NileChat-3B


config.json:   0%|          | 0.00/781 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.25k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/435 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]


Loaded successfully.
Model: UBC-NLP/NileChat-3B
load_in_4bit: True
model device: cuda:0
model dtype: torch.float16
tokenizer type: <class 'transformers.models.qwen2.tokenization_qwen2.Qwen2Tokenizer'>
vocab size: 151665
pad token: <|endoftext|>
pad token id: 151643
eos token: <|im_end|>
eos token id: 151645
has chat_template: True

NileChat chat template is available and will be used in Cell 11.


### **Apply Chat Template**

In [16]:
# ============================================================
# Cell 11 — Build ntkmirror examples for NileChat
# Uses NileChat chat template when available
# ============================================================

from ntkmirror import Example
import pandas as pd

SYSTEM_MARKER = "### System:"
INSTRUCTION_MARKER = "### Instruction:"
RESPONSE_MARKER = "### Arabic translation:"

def get_message_content(messages, role):
    for m in messages:
        if m.get("role") == role:
            return m.get("content", "")
    return ""

def format_manual_ntkmirror_prompt(system_text, user_text):
    """
    Fallback prompt format only if tokenizer has no chat template.
    """
    return (
        f"{SYSTEM_MARKER}\n"
        f"{system_text.strip()}\n\n"
        f"{INSTRUCTION_MARKER}\n"
        f"{user_text.strip()}\n\n"
        f"{RESPONSE_MARKER}\n"
    )

def build_nilechat_prompt(system_text, user_text):
    """
    Prompt only.
    Prefer tokenizer.apply_chat_template because NileChat is an instructed chat model.
    """
    system_text = str(system_text).strip()
    user_text = str(user_text).strip()

    if tokenizer.chat_template is not None:
        messages = []

        # Many Qwen-style templates support system role.
        # If a tokenizer rejects system messages, fallback below catches it.
        if system_text:
            messages.append({"role": "system", "content": system_text})

        messages.append({"role": "user", "content": user_text})

        try:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
        except Exception as e:
            print("WARNING: apply_chat_template with system role failed once.")
            print("Reason:", repr(e))
            print("Falling back to user-only chat template.")

            combined_user = (
                f"{system_text}\n\n{user_text}"
                if system_text
                else user_text
            )

            return tokenizer.apply_chat_template(
                [{"role": "user", "content": combined_user}],
                tokenize=False,
                add_generation_prompt=True,
            )

    return format_manual_ntkmirror_prompt(system_text, user_text)

def format_sft_text(system_text, user_text, assistant_text=None, add_eos=True):
    """
    Kept for inference compatibility.
    For NileChat, this uses the chat template for prompt construction.
    """
    text = build_nilechat_prompt(system_text, user_text)

    if assistant_text is not None:
        text += str(assistant_text).strip()
        if add_eos and tokenizer.eos_token is not None:
            text += tokenizer.eos_token

    return text

def row_to_ntkmirror_example(row):
    messages = row["messages"]

    system_text = get_message_content(messages, "system")
    user_text = get_message_content(messages, "user")
    assistant_text = get_message_content(messages, "assistant")

    prompt = build_nilechat_prompt(system_text, user_text)

    completion = str(assistant_text).strip()
    if tokenizer.eos_token is not None:
        completion += tokenizer.eos_token

    return Example(prompt=prompt, completion=completion)

train_ntk_examples = [
    row_to_ntkmirror_example(train_dataset[i])
    for i in range(len(train_dataset))
]

eval_ntk_examples = [
    row_to_ntkmirror_example(eval_dataset[i])
    for i in range(len(eval_dataset))
]

print("ntkmirror train examples:", len(train_ntk_examples))
print("ntkmirror eval examples:", len(eval_ntk_examples))

print("\nFormatted training example:")
print(train_ntk_examples[0].prompt + train_ntk_examples[0].completion)

# ------------------------------------------------------------
# Token length diagnostics
# ------------------------------------------------------------

def count_example_tokens(ex):
    ids = tokenizer(
        ex.prompt + ex.completion,
        add_special_tokens=False,
        truncation=False,
    )["input_ids"]
    return len(ids)

train_lengths = [count_example_tokens(ex) for ex in train_ntk_examples]
eval_lengths = [count_example_tokens(ex) for ex in eval_ntk_examples]

def summarize_lengths(lengths, name):
    s = pd.Series(lengths)
    print(f"\n{name} token length summary:")
    print("count:", len(s))
    print("min:", int(s.min()))
    print("median:", int(s.median()))
    print("p90:", int(s.quantile(0.90)))
    print("p95:", int(s.quantile(0.95)))
    print("p99:", int(s.quantile(0.99)))
    print("max:", int(s.max()))

summarize_lengths(train_lengths, "Train")
summarize_lengths(eval_lengths, "Eval")

too_long_train = sum(x > MAX_SEQ_LENGTH for x in train_lengths)
too_long_eval = sum(x > MAX_SEQ_LENGTH for x in eval_lengths)

print("\nMAX_SEQ_LENGTH:", MAX_SEQ_LENGTH)
print("Train examples longer than MAX_SEQ_LENGTH:", too_long_train)
print("Eval examples longer than MAX_SEQ_LENGTH:", too_long_eval)

if too_long_train > 0 or too_long_eval > 0:
    print("\nWARNING: Some examples are longer than MAX_SEQ_LENGTH.")
    print("Recommended first fix: reduce MAX_FEW_SHOT_EXAMPLE_CHARS.")
else:
    print("\nOK: MAX_SEQ_LENGTH is enough.")

ntkmirror train examples: 3108
ntkmirror eval examples: 1118

Formatted training example:
<|im_start|>system
You are a professional machine translation system. Translate the current English dialogue turn into natural dialectal Arabic. Use the provided training examples only as style and dialect guidance. Return only the translation, without explanation.<|im_end|>
<|im_start|>user
Task:
Translate the current English dialogue turn into the target dialectal Arabic variety.

Few-shot training examples:
Example 1 (config=EG, dialect=Egyptian Arabic (Cairene) Dialect, domain=Agriculture and farming)
English:
By monitoring, we'll know exactly when the pest levels are high enough to justify spraying. This saves you money and protects the environment.

Arabic:
لما نراقب، هنعرف بالضبط امتى مستويات الحشرات عالية كفاية عشان نبرر الرش. ده بيوفر فلوس وبيحمي البيئة.

Example 2 (config=EG, dialect=Egyptian Arabic (Cairene) Dialect, domain=Agriculture and farming)
English:
So, what will this cost me fo

#### **Initialize ntkmirror tuner**

In [17]:
# ============================================================
# Cell 12 — Initialize ntkmirror ForwardFineTuner
# ============================================================

from ntkmirror import ForwardFineTuner

tuner = ForwardFineTuner(
    model=model,
    tokenizer=tokenizer,
    gates=NTK_GATES,
    layers=NTK_LAYERS,
    max_log_gate=NTK_MAX_LOG_GATE,
    hook_site=NTK_HOOK_SITE,
)

print("ForwardFineTuner ready.")
print("Layer path:", tuner.layer_path)
print("Number of decoder layers:", len(tuner.decoder_layers))
print("Hidden size:", tuner.hidden_size)
print("Selected layer ids:", tuner.layer_ids[:10], "..." if len(tuner.layer_ids) > 10 else "")

ForwardFineTuner ready.
Layer path: model.layers
Number of decoder layers: 36
Hidden size: 2048
Selected layer ids: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] ...


#### Check for existing checkpoints

In [18]:
# ============================================================
# Cell 13 — Check for existing ntkmirror checkpoints
# ============================================================

from pathlib import Path
import json
import re

def checkpoint_step(path):
    m = re.search(r"checkpoint-(\d+)", str(path))
    return int(m.group(1)) if m else -1

def get_last_ntk_checkpoint(output_dir):
    output_dir = Path(output_dir)
    checkpoints = sorted(
        [p for p in output_dir.glob("checkpoint-*") if p.is_dir()],
        key=checkpoint_step,
    )
    return checkpoints[-1] if checkpoints else None

last_checkpoint = get_last_ntk_checkpoint(OUTPUT_DIR)

if last_checkpoint:
    print("Found ntkmirror checkpoint:")
    print(last_checkpoint)
else:
    print("No ntkmirror checkpoint found. Training will start from scratch.")

No ntkmirror checkpoint found. Training will start from scratch.


### **Build optimizer, scheduler, and checkpoint helpers**

In [19]:
# ============================================================
# Cell 14 — Build ntkmirror optimizer/scheduler/checkpoint helpers
# ============================================================

import math
import json
import shutil
from pathlib import Path

import torch
from transformers import get_cosine_schedule_with_warmup

from ntkmirror.data import batches, make_batch
from ntkmirror.losses import causal_loss_from_logits, token_accuracy_from_logits

device = next(model.parameters()).device

num_micro_batches_per_epoch = math.ceil(len(train_ntk_examples) / PER_DEVICE_BATCH_SIZE)
TOTAL_OPTIMIZER_STEPS = math.ceil(
    num_micro_batches_per_epoch * NUM_EPOCHS / GRAD_ACCUM_STEPS
)
WARMUP_STEPS = int(TOTAL_OPTIMIZER_STEPS * WARMUP_RATIO)

print("Micro-batches per epoch:", num_micro_batches_per_epoch)
print("Total optimizer steps:", TOTAL_OPTIMIZER_STEPS)
print("Warmup steps:", WARMUP_STEPS)

def evaluate_ntk_nll(examples, batch_size=1, max_length=MAX_SEQ_LENGTH, limit=None):
    if limit is not None:
        examples = examples[: int(limit)]

    return tuner.evaluate_nll(
        examples,
        batch_size=batch_size,
        max_length=max_length,
        use_controller=True,
    )

def save_ntk_checkpoint(
    checkpoint_dir,
    global_step,
    epoch,
    optimizer,
    scheduler,
    log_history,
    best_metric,
    best_model_checkpoint,
):
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    controller_path = checkpoint_dir / "controller.pt"
    optimizer_path = checkpoint_dir / "optimizer.pt"
    scheduler_path = checkpoint_dir / "scheduler.pt"
    state_path = checkpoint_dir / "trainer_state.json"

    tuner.save(controller_path)

    torch.save(
        {
            "optimizer": optimizer.state_dict(),
            "global_step": global_step,
            "epoch": epoch,
        },
        optimizer_path,
    )

    torch.save(
        {
            "scheduler": scheduler.state_dict(),
            "global_step": global_step,
            "epoch": epoch,
        },
        scheduler_path,
    )

    state = {
        "global_step": int(global_step),
        "epoch": float(epoch),
        "best_metric": None if best_metric is None else float(best_metric),
        "best_model_checkpoint": None if best_model_checkpoint is None else str(best_model_checkpoint),
        "log_history": log_history,
        "total_optimizer_steps": int(TOTAL_OPTIMIZER_STEPS),
        "num_train_epochs": int(NUM_EPOCHS),
        "learning_rate": float(LEARNING_RATE),
        "per_device_train_batch_size": int(PER_DEVICE_BATCH_SIZE),
        "gradient_accumulation_steps": int(GRAD_ACCUM_STEPS),
        "ntk_gates": int(NTK_GATES),
        "ntk_layers": str(NTK_LAYERS),
        "ntk_max_log_gate": float(NTK_MAX_LOG_GATE),
        "ntk_hook_site": str(NTK_HOOK_SITE),
    }

    state_path.write_text(json.dumps(state, indent=2), encoding="utf-8")

def load_ntk_checkpoint(checkpoint_dir):
    checkpoint_dir = Path(checkpoint_dir)

    controller_path = checkpoint_dir / "controller.pt"
    optimizer_path = checkpoint_dir / "optimizer.pt"
    scheduler_path = checkpoint_dir / "scheduler.pt"
    state_path = checkpoint_dir / "trainer_state.json"

    if not controller_path.exists():
        raise FileNotFoundError(controller_path)

    tuner.load(controller_path)

    state = json.loads(state_path.read_text(encoding="utf-8")) if state_path.exists() else {}
    opt_state = torch.load(optimizer_path, map_location="cpu") if optimizer_path.exists() else None
    sch_state = torch.load(scheduler_path, map_location="cpu") if scheduler_path.exists() else None

    return state, opt_state, sch_state

def prune_old_checkpoints(output_dir, save_total_limit, best_model_checkpoint=None):
    output_dir = Path(output_dir)

    checkpoints = sorted(
        [p for p in output_dir.glob("checkpoint-*") if p.is_dir()],
        key=checkpoint_step,
    )

    if save_total_limit is None or save_total_limit <= 0:
        return

    best_model_checkpoint = str(best_model_checkpoint) if best_model_checkpoint else None

    while len(checkpoints) > save_total_limit:
        candidate = checkpoints.pop(0)

        if best_model_checkpoint and str(candidate) == best_model_checkpoint:
            checkpoints.append(candidate)
            checkpoints = sorted(checkpoints, key=checkpoint_step)

            # If the best checkpoint is the oldest, delete the next oldest instead.
            non_best = [p for p in checkpoints if str(p) != best_model_checkpoint]
            if not non_best:
                break

            candidate = non_best[0]
            checkpoints.remove(candidate)

        print("Pruning old checkpoint:", candidate)
        shutil.rmtree(candidate, ignore_errors=True)

print("Checkpoint helpers ready.")

Micro-batches per epoch: 3108
Total optimizer steps: 3885
Warmup steps: 116
Checkpoint helpers ready.


### **Initialize controller and optimizer state**

In [20]:
# ============================================================
# Cell 15 — Initialize or resume ntkmirror controller
# Stronger run: random representative gate-scoring subset
# ============================================================

import math
import random
import torch

if last_checkpoint:
    print("Resuming from:", last_checkpoint)

    saved_state, opt_state, sch_state = load_ntk_checkpoint(last_checkpoint)

    START_GLOBAL_STEP = int(saved_state.get("global_step", checkpoint_step(last_checkpoint)))
    START_EPOCH_FLOAT = float(saved_state.get("epoch", 0.0))

    log_history = list(saved_state.get("log_history", []))

    BEST_EVAL_LOSS_SO_FAR = saved_state.get("best_metric", None)
    BEST_CHECKPOINT_SO_FAR = saved_state.get("best_model_checkpoint", None)

    print("Loaded controller from checkpoint.")
    print("Start global step:", START_GLOBAL_STEP)
    print("Best eval loss so far:", BEST_EVAL_LOSS_SO_FAR)
    print("Best checkpoint so far:", BEST_CHECKPOINT_SO_FAR)

else:
    print("No checkpoint found for this stronger NTK experiment.")
    print("Starting fresh controller initialization.")

    # --------------------------------------------------------
    # Random representative scoring subset
    # --------------------------------------------------------

    rng = random.Random(SEED)

    score_k = min(int(NTK_SCORE_EXAMPLES), len(train_ntk_examples))

    scoring_indices = rng.sample(
        range(len(train_ntk_examples)),
        k=score_k,
    )

    score_ntk_examples = [train_ntk_examples[i] for i in scoring_indices]

    effective_score_batches = math.ceil(
        len(score_ntk_examples) / PER_DEVICE_BATCH_SIZE
    )

    print("\nGate scoring setup:")
    print("  total train examples:", len(train_ntk_examples))
    print("  scoring examples:", len(score_ntk_examples))
    print("  per-device batch size:", PER_DEVICE_BATCH_SIZE)
    print("  effective score batches:", effective_score_batches)

    print("\nInitializing ntkmirror controller by activation-gradient gate scoring...")

    init_stats = tuner.initialize_controller(
        score_ntk_examples,
        score_batches=effective_score_batches,
        batch_size=PER_DEVICE_BATCH_SIZE,
        max_length=MAX_SEQ_LENGTH,
    )

    print("\nController initialization stats:")
    print(init_stats)

    START_GLOBAL_STEP = 0
    START_EPOCH_FLOAT = 0.0
    log_history = []
    BEST_EVAL_LOSS_SO_FAR = None
    BEST_CHECKPOINT_SO_FAR = None
    opt_state = None
    sch_state = None

if tuner.controller is None:
    raise RuntimeError("ntkmirror controller was not initialized or loaded.")

controller_gate_count = int(tuner.controller.raw.numel())

print("\nController check:")
print("  expected gates:", NTK_GATES)
print("  actual gates:", controller_gate_count)
print("  max_log_gate:", tuner.controller.max_log_gate)
print("  hook_site:", tuner.controller.hook_site)

if controller_gate_count != int(NTK_GATES):
    raise RuntimeError(
        f"Loaded controller has {controller_gate_count} gates, "
        f"but current config expects {NTK_GATES}. "
        "This usually means you are accidentally resuming an incompatible experiment."
    )

# ------------------------------------------------------------
# Optimizer and scheduler
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    [tuner.controller.raw],
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=TOTAL_OPTIMIZER_STEPS,
)

if opt_state is not None:
    optimizer.load_state_dict(opt_state["optimizer"])
    print("\nLoaded optimizer state.")

if sch_state is not None:
    scheduler.load_state_dict(sch_state["scheduler"])
    print("Loaded scheduler state.")

print("\nTraining state:")
print("  start global step:", START_GLOBAL_STEP)
print("  total optimizer steps:", TOTAL_OPTIMIZER_STEPS)
print("  warmup steps:", WARMUP_STEPS)
print("  learning rate:", LEARNING_RATE)
print("  best eval loss so far:", BEST_EVAL_LOSS_SO_FAR)
print("  best checkpoint so far:", BEST_CHECKPOINT_SO_FAR)

No checkpoint found for this stronger NTK experiment.
Starting fresh controller initialization.

Gate scoring setup:
  total train examples: 3108
  scoring examples: 256
  per-device batch size: 1
  effective score batches: 256

Initializing ntkmirror controller by activation-gradient gate scoring...

Controller initialization stats:
{'selected_gates': 10000.0, 'select_seconds': 168.64292169100008, 'hook_site': 'layer_output'}

Controller check:
  expected gates: 10000
  actual gates: 10000
  max_log_gate: 0.1
  hook_site: layer_output

Training state:
  start global step: 0
  total optimizer steps: 3885
  warmup steps: 116
  learning rate: 0.001
  best eval loss so far: None
  best checkpoint so far: None


### **Training**

In [ ]:
# ============================================================
# Cell 16 — Train or resume ntkmirror controller
# ============================================================

import time
import random
from tqdm.auto import tqdm

model.eval()

global_step = int(START_GLOBAL_STEP)
micro_step_seen = global_step * GRAD_ACCUM_STEPS

best_eval_loss = BEST_EVAL_LOSS_SO_FAR
best_model_checkpoint = BEST_CHECKPOINT_SO_FAR

running_loss = 0.0
running_items = 0

train_start = time.time()

print("Starting ntkmirror training.")
print("Starting global step:", global_step)
print("Target optimizer steps:", TOTAL_OPTIMIZER_STEPS)

optimizer.zero_grad(set_to_none=True)

stop_training = False

for epoch in range(NUM_EPOCHS):
    epoch_indices = list(range(len(train_ntk_examples)))
    rng = random.Random(SEED + epoch)
    rng.shuffle(epoch_indices)

    epoch_examples = [train_ntk_examples[i] for i in epoch_indices]
    epoch_batches = list(batches(epoch_examples, PER_DEVICE_BATCH_SIZE))

    for micro_batch_index, chunk in enumerate(tqdm(epoch_batches, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")):
        current_micro_index = epoch * len(epoch_batches) + micro_batch_index

        if current_micro_index < micro_step_seen:
            continue

        batch = make_batch(
            tokenizer,
            chunk,
            device=device,
            max_length=MAX_SEQ_LENGTH,
        )

        tuner.controller.attach()

        loss = tuner._loss(batch)

        if NTK_L2 > 0:
            loss = loss + float(NTK_L2) * tuner.controller.s.float().pow(2).mean()

        scaled_loss = loss / GRAD_ACCUM_STEPS
        scaled_loss.backward()

        running_loss += float(loss.detach().item())
        running_items += 1

        should_step = ((current_micro_index + 1) % GRAD_ACCUM_STEPS == 0)

        if should_step:
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

            global_step += 1
            epoch_float = epoch + (micro_batch_index + 1) / max(1, len(epoch_batches))

            if global_step % LOGGING_STEPS == 0 or global_step == 1:
                avg_loss = running_loss / max(1, running_items)
                current_lr = scheduler.get_last_lr()[0]

                row = {
                    "loss": float(avg_loss),
                    "learning_rate": float(current_lr),
                    "epoch": float(epoch_float),
                    "step": int(global_step),
                }

                log_history.append(row)

                print(
                    f"step {global_step}/{TOTAL_OPTIMIZER_STEPS} | "
                    f"epoch {epoch_float:.3f} | "
                    f"loss {avg_loss:.6f} | "
                    f"lr {current_lr:.3e}"
                )

                running_loss = 0.0
                running_items = 0

            if global_step % EVAL_STEPS == 0 or global_step == TOTAL_OPTIMIZER_STEPS:
                tuner.controller.remove()

                eval_metrics = evaluate_ntk_nll(
                    eval_ntk_examples,
                    batch_size=1,
                    max_length=MAX_SEQ_LENGTH,
                    limit=EVAL_NLL_LIMIT,
                )

                eval_loss = float(eval_metrics["nll"])
                eval_token_acc = float(eval_metrics["token_acc"])

                row = {
                    "eval_loss": eval_loss,
                    "eval_token_acc": eval_token_acc,
                    "eval_tokens": float(eval_metrics["tokens"]),
                    "epoch": float(epoch_float),
                    "step": int(global_step),
                }

                log_history.append(row)

                print(
                    f"\nEVAL step {global_step}: "
                    f"eval_loss={eval_loss:.6f}, "
                    f"token_acc={eval_token_acc:.6f}, "
                    f"tokens={eval_metrics['tokens']:.0f}\n"
                )

                if best_eval_loss is None or eval_loss < float(best_eval_loss):
                    best_eval_loss = eval_loss
                    best_model_checkpoint = str(OUTPUT_DIR / f"checkpoint-{global_step}")
                    print("New best checkpoint:", best_model_checkpoint)

                tuner.controller.attach()

            if global_step % SAVE_STEPS == 0 or global_step == TOTAL_OPTIMIZER_STEPS:
                tuner.controller.remove()

                ckpt_dir = OUTPUT_DIR / f"checkpoint-{global_step}"

                save_ntk_checkpoint(
                    ckpt_dir,
                    global_step=global_step,
                    epoch=epoch_float,
                    optimizer=optimizer,
                    scheduler=scheduler,
                    log_history=log_history,
                    best_metric=best_eval_loss,
                    best_model_checkpoint=best_model_checkpoint,
                )

                print("Saved checkpoint:", ckpt_dir)

                prune_old_checkpoints(
                    OUTPUT_DIR,
                    SAVE_TOTAL_LIMIT,
                    best_model_checkpoint=best_model_checkpoint,
                )

                tuner.controller.attach()

            if global_step >= TOTAL_OPTIMIZER_STEPS:
                stop_training = True
                break

    if stop_training:
        break

tuner.controller.remove()

elapsed = time.time() - train_start

trainer_stats = {
    "global_step": global_step,
    "best_eval_loss": best_eval_loss,
    "best_model_checkpoint": best_model_checkpoint,
    "train_seconds": elapsed,
}

print("\nTraining finished.")
print(json.dumps(trainer_stats, indent=2))

Starting ntkmirror training.
Starting global step: 0
Target optimizer steps: 3885


Epoch 1/10:   0%|          | 0/3108 [00:00<?, ?it/s]

step 1/3885 | epoch 0.003 | loss 1.291213 | lr 8.621e-06
step 10/3885 | epoch 0.026 | loss 1.328794 | lr 8.621e-05
step 20/3885 | epoch 0.051 | loss 1.418725 | lr 1.724e-04
step 30/3885 | epoch 0.077 | loss 1.449872 | lr 2.586e-04
step 40/3885 | epoch 0.103 | loss 1.360086 | lr 3.448e-04
step 50/3885 | epoch 0.129 | loss 1.371135 | lr 4.310e-04
step 60/3885 | epoch 0.154 | loss 1.423270 | lr 5.172e-04
step 70/3885 | epoch 0.180 | loss 1.416307 | lr 6.034e-04
step 80/3885 | epoch 0.206 | loss 1.395128 | lr 6.897e-04
step 90/3885 | epoch 0.232 | loss 1.427206 | lr 7.759e-04
step 100/3885 | epoch 0.257 | loss 1.390051 | lr 8.621e-04

EVAL step 100: eval_loss=1.469086, token_acc=0.666891, tokens=38717

New best checkpoint: /content/drive/MyDrive/alexandria_qwen35_sft/runs/nilechat3b_alexandria_eg_only_context3_complete2shot_ntkmirror_g10000_score256_mlgate010_lr1e3_all_10epochs/checkpoint-100
Saved checkpoint: /content/drive/MyDrive/alexandria_qwen35_sft/runs/nilechat3b_alexandria_eg_only_

Epoch 2/10:   0%|          | 0/3108 [00:00<?, ?it/s]

step 390/3885 | epoch 1.004 | loss 1.249753 | lr 9.870e-04
step 400/3885 | epoch 1.030 | loss 1.266093 | lr 9.861e-04

EVAL step 400: eval_loss=1.379980, token_acc=0.674019, tokens=38717

New best checkpoint: /content/drive/MyDrive/alexandria_qwen35_sft/runs/nilechat3b_alexandria_eg_only_context3_complete2shot_ntkmirror_g10000_score256_mlgate010_lr1e3_all_10epochs/checkpoint-400
Saved checkpoint: /content/drive/MyDrive/alexandria_qwen35_sft/runs/nilechat3b_alexandria_eg_only_context3_complete2shot_ntkmirror_g10000_score256_mlgate010_lr1e3_all_10epochs/checkpoint-400
step 410/3885 | epoch 1.055 | loss 1.334679 | lr 9.851e-04
step 420/3885 | epoch 1.081 | loss 1.234214 | lr 9.840e-04
step 430/3885 | epoch 1.107 | loss 1.274993 | lr 9.830e-04
step 440/3885 | epoch 1.133 | loss 1.273869 | lr 9.819e-04
step 450/3885 | epoch 1.158 | loss 1.306536 | lr 9.807e-04
step 460/3885 | epoch 1.184 | loss 1.364494 | lr 9.796e-04
step 470/3885 | epoch 1.210 | loss 1.235248 | lr 9.784e-04
step 480/3885 

...

step 780/3885 | epoch 2.008 | loss 1.369050 | lr 9.254e-04
step 790/3885 | epoch 2.033 | loss 1.272769 | lr 9.231e-04
step 800/3885 | epoch 2.059 | loss 1.180678 | lr 9.209e-04

EVAL step 800: eval_loss=1.344777, token_acc=0.677274, tokens=38717

New best checkpoint: /content/drive/MyDrive/alexandria_qwen35_sft/runs/nilechat3b_alexandria_eg_only_context3_complete2shot_ntkmirror_g10000_score256_mlgate010_lr1e3_all_10epochs/checkpoint-800
Saved checkpoint: /content/drive/MyDrive/alexandria_qwen35_sft/runs/nilechat3b_alexandria_eg_only_context3_complete2shot_ntkmirror_g10000_score256_mlgate010_lr1e3_all_10epochs/checkpoint-800
step 810/3885 | epoch 2.085 | loss 1.210173 | lr 9.186e-04
step 820/3885 | epoch 2.111 | loss 1.278650 | lr 9.164e-04
step 830/3885 | epoch 2.136 | loss 1.263582 | lr 9.140e-04
step 840/3885 | epoch 2.162 | loss 1.289063 | lr 9.117e-04
step 850/3885 | epoch 2.188 | loss 1.348924 | lr 9.093e-04
step 860/3885 | epoch 2.214 | loss 1.271689 | lr 9.069e-04
step 870/3885 

Epoch 4/10:   0%|          | 0/3108 [00:00<?, ?it/s]

step 1170/3885 | epoch 3.012 | loss 1.255274 | lr 8.191e-04
step 1180/3885 | epoch 3.037 | loss 1.344039 | lr 8.159e-04
step 1190/3885 | epoch 3.063 | loss 1.282922 | lr 8.127e-04
step 1200/3885 | epoch 3.089 | loss 1.196193 | lr 8.094e-04

EVAL step 1200: eval_loss=1.329571, token_acc=0.678952, tokens=38717

New best checkpoint: /content/drive/MyDrive/alexandria_qwen35_sft/runs/nilechat3b_alexandria_eg_only_context3_complete2shot_ntkmirror_g10000_score256_mlgate010_lr1e3_all_10epochs/checkpoint-1200
Saved checkpoint: /content/drive/MyDrive/alexandria_qwen35_sft/runs/nilechat3b_alexandria_eg_only_context3_complete2shot_ntkmirror_g10000_score256_mlgate010_lr1e3_all_10epochs/checkpoint-1200
step 1210/3885 | epoch 3.115 | loss 1.195677 | lr 8.061e-04
step 1220/3885 | epoch 3.140 | loss 1.273923 | lr 8.028e-04
step 1230/3885 | epoch 3.166 | loss 1.302003 | lr 7.995e-04
step 1240/3885 | epoch 3.192 | loss 1.234936 | lr 7.961e-04
step 1250/3885 | epoch 3.218 | loss 1.227985 | lr 7.928e-04
st

Epoch 5/10:   0%|          | 0/3108 [00:00<?, ?it/s]

step 1560/3885 | epoch 4.015 | loss 1.250646 | lr 6.795e-04
step 1570/3885 | epoch 4.041 | loss 1.175156 | lr 6.756e-04
step 1580/3885 | epoch 4.067 | loss 1.230612 | lr 6.717e-04
step 1590/3885 | epoch 4.093 | loss 1.182574 | lr 6.678e-04
step 1600/3885 | epoch 4.118 | loss 1.191106 | lr 6.638e-04

EVAL step 1600: eval_loss=1.321284, token_acc=0.679856, tokens=38717

New best checkpoint: /content/drive/MyDrive/alexandria_qwen35_sft/runs/nilechat3b_alexandria_eg_only_context3_complete2shot_ntkmirror_g10000_score256_mlgate010_lr1e3_all_10epochs/checkpoint-1600
Saved checkpoint: /content/drive/MyDrive/alexandria_qwen35_sft/runs/nilechat3b_alexandria_eg_only_context3_complete2shot_ntkmirror_g10000_score256_mlgate010_lr1e3_all_10epochs/checkpoint-1600
step 1610/3885 | epoch 4.144 | loss 1.262802 | lr 6.599e-04
step 1620/3885 | epoch 4.170 | loss 1.209257 | lr 6.559e-04
step 1630/3885 | epoch 4.196 | loss 1.202058 | lr 6.520e-04
step 1640/3885 | epoch 4.221 | loss 1.136249 | lr 6.480e-04
st

### **Find, load, and save BEST controller**

In [ ]:
# ============================================================
# Cell 17 — Find, load, and save BEST ntkmirror controller
# IMPORTANT:
# ============================================================

from pathlib import Path
import json

def find_best_ntk_checkpoint_from_logs(output_dir):
    output_dir = Path(output_dir)

    state_files = list(output_dir.rglob("trainer_state.json"))
    if not state_files:
        raise FileNotFoundError(f"No trainer_state.json found under: {output_dir}")

    all_eval_rows = []

    for sf in state_files:
        try:
            state = json.loads(sf.read_text(encoding="utf-8"))

            # First collect eval rows
            for row in state.get("log_history", []):
                if "eval_loss" in row and "step" in row:
                    all_eval_rows.append({
                        "step": int(row["step"]),
                        "eval_loss": float(row["eval_loss"]),
                        "epoch": row.get("epoch", None),
                        "state_file": sf,
                    })

            # Also respect stored best checkpoint if available
            best_ckpt = state.get("best_model_checkpoint", None)
            best_metric = state.get("best_metric", None)

            if best_ckpt is not None and best_metric is not None:
                best_path = Path(best_ckpt)

                if not best_path.exists():
                    best_path = output_dir / best_path.name

                if best_path.exists():
                    step = checkpoint_step(best_path)

                    epoch = None
                    for row in state.get("log_history", []):
                        if int(row.get("step", -1)) == int(step) and "eval_loss" in row:
                            epoch = row.get("epoch", None)
                            break

                    all_eval_rows.append({
                        "step": int(step),
                        "eval_loss": float(best_metric),
                        "epoch": epoch,
                        "state_file": sf,
                    })

        except Exception as e:
            print(f"Skipping bad trainer_state file: {sf}")
            print("Reason:", repr(e))

    if not all_eval_rows:
        raise RuntimeError(
            "No eval_loss records found. "
            "You need at least one completed eval/save step before selecting the best checkpoint."
        )

    best_row = min(all_eval_rows, key=lambda x: x["eval_loss"])

    best_step = int(best_row["step"])
    best_metric = float(best_row["eval_loss"])
    best_epoch = best_row["epoch"]
    best_state_file = best_row["state_file"]

    best_checkpoint = output_dir / f"checkpoint-{best_step}"

    if not best_checkpoint.exists():
        existing = sorted([p.name for p in output_dir.glob("checkpoint-*")])
        raise FileNotFoundError(
            f"Best checkpoint is missing: {best_checkpoint}\n"
            f"Best step from logs = {best_step}, eval_loss = {best_metric}\n"
            f"Existing checkpoints: {existing}\n"
            f"Keep SAVE_TOTAL_LIMIT high enough."
        )

    controller_file = best_checkpoint / "controller.pt"

    if not controller_file.exists():
        raise FileNotFoundError(
            f"Best checkpoint exists but controller.pt is missing:\n{controller_file}"
        )

    return best_checkpoint, best_step, best_metric, best_epoch, best_state_file


BEST_CHECKPOINT_PATH, BEST_STEP, BEST_EVAL_LOSS, BEST_EPOCH, BEST_STATE_FILE = find_best_ntk_checkpoint_from_logs(OUTPUT_DIR)

print("Best ntkmirror checkpoint:")
print("  path:", BEST_CHECKPOINT_PATH)
print("  step:", BEST_STEP)
print("  epoch:", BEST_EPOCH)
print("  eval_loss:", BEST_EVAL_LOSS)
print("  trainer_state:", BEST_STATE_FILE)

print("\nLoading best ntkmirror controller...")
tuner.load(BEST_CHECKPOINT_PATH / "controller.pt")
print("Loaded best controller into tuner.")

BEST_CONTROLLER_PATH = CONTROLLER_DIR / f"{EXPERIMENT_NAME}_best_step{BEST_STEP}.pt"
BEST_MANIFEST_PATH = CONTROLLER_DIR / f"{EXPERIMENT_NAME}_best_step{BEST_STEP}.manifest.json"

tuner.save(BEST_CONTROLLER_PATH)
tuner.write_manifest(BEST_MANIFEST_PATH)

print("\nSaved BEST controller to:")
print(BEST_CONTROLLER_PATH)

print("\nSaved BEST manifest to:")
print(BEST_MANIFEST_PATH)

print("\nActive setup is now:")
print("  base model + BEST ntkmirror controller")
print("  BEST_CHECKPOINT_PATH:", BEST_CHECKPOINT_PATH)
print("  BEST_CONTROLLER_PATH:", BEST_CONTROLLER_PATH)
print("  BEST_STEP:", BEST_STEP)
print("  BEST_EVAL_LOSS:", BEST_EVAL_LOSS)

Best ntkmirror checkpoint:
  path: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_alexandria_eg_only_context3_complete2shot_ntkmirror_g10000_score256_mlgate010_lr1e3_all_10epochs/checkpoint-1300
  step: 1300
  epoch: 3.346203346203346
  eval_loss: 2.4269274191662777
  trainer_state: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_alexandria_eg_only_context3_complete2shot_ntkmirror_g10000_score256_mlgate010_lr1e3_all_10epochs/checkpoint-1300/trainer_state.json

Loading best ntkmirror controller...
Loaded best controller into tuner.

Saved BEST controller to:
/content/drive/MyDrive/alexandria_qwen35_sft/final_adapters/qwen3_4b_alexandria_eg_only_context3_complete2shot_ntkmirror_g10000_score256_mlgate010_lr1e3_all_10epochs/qwen3_4b_alexandria_eg_only_context3_complete2shot_ntkmirror_g10000_score256_mlgate010_lr1e3_all_10epochs_best_step1300.pt

Saved BEST manifest to:
/content/drive/MyDrive/alexandria_qwen35_sft/final_adapters/qwen3_4b_alexandria_eg_only_context3_co

### **Quick Inference**

In [ ]:
# ============================================================
# Cell 18 — Quick inference function
# Uses BEST ntkmirror controller loaded in Cell 17
# NileChat-compatible chat-template prompting
# ============================================================

import torch

if "BEST_CONTROLLER_PATH" not in globals():
    raise RuntimeError(
        "BEST_CONTROLLER_PATH is not defined. "
        "Run Cell 17 first to load the best controller."
    )

print("Inference will use BEST ntkmirror controller:")
print("BEST_CHECKPOINT_PATH:", BEST_CHECKPOINT_PATH)
print("BEST_CONTROLLER_PATH:", BEST_CONTROLLER_PATH)
print("BEST_STEP:", BEST_STEP)
print("BEST_EVAL_LOSS:", BEST_EVAL_LOSS)

model.eval()

def extract_assistant_answer(decoded_text, prompt_text=None):
    """
    Robust extraction for chat-template models.
    Prefer removing the prompt prefix if possible.
    Then strip common special tokens and role markers.
    """

    answer = decoded_text

    if prompt_text is not None and decoded_text.startswith(prompt_text):
        answer = decoded_text[len(prompt_text):]

    # Fallbacks for manual / chat markers
    possible_markers = [
        RESPONSE_MARKER,
        "<|assistant|>",
        "assistant\n",
        "assistant:",
        "Assistant:",
    ]

    for marker in possible_markers:
        if marker in answer:
            answer = answer.split(marker)[-1]

    special_tokens = [
        tokenizer.eos_token,
        tokenizer.pad_token,
        "<|endoftext|>",
        "<|im_end|>",
        "<|end|>",
        "<|assistant|>",
        "<|user|>",
        "<|system|>",
    ]

    for tok in special_tokens:
        if tok:
            answer = answer.replace(tok, "")

    return answer.strip()

def generate_translation_from_row(row, max_new_tokens=120):
    user_text = make_user_prompt(row)

    prompt = format_sft_text(
        system_text=SYSTEM_PROMPT,
        user_text=user_text,
        assistant_text=None,
        add_eos=False,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        add_special_tokens=False,
    ).to(model.device)

    tuner.controller.attach()

    try:
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                repetition_penalty=1.05,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
            )
    finally:
        tuner.controller.remove()

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)
    answer = extract_assistant_answer(decoded, prompt_text=prompt)

    return answer, decoded

sample = eval_df.sample(1, random_state=SEED).iloc[0].to_dict()

pred, raw = generate_translation_from_row(sample)

print("Country/config:", sample["config"])
print("Dialect:", sample["dialect"])
print("Domain:", sample["domain"])
print("\nEnglish:")
print(sample["source_text"])
print("\nReference Arabic:")
print(sample["target_arabic"])
print("\nPrediction:")
print(pred)

Inference will use BEST ntkmirror controller:
BEST_CHECKPOINT_PATH: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_alexandria_eg_only_context3_complete2shot_ntkmirror_g10000_score256_mlgate010_lr1e3_all_10epochs/checkpoint-1300
BEST_CONTROLLER_PATH: /content/drive/MyDrive/alexandria_qwen35_sft/final_adapters/qwen3_4b_alexandria_eg_only_context3_complete2shot_ntkmirror_g10000_score256_mlgate010_lr1e3_all_10epochs/qwen3_4b_alexandria_eg_only_context3_complete2shot_ntkmirror_g10000_score256_mlgate010_lr1e3_all_10epochs_best_step1300.pt
BEST_STEP: 1300
BEST_EVAL_LOSS: 2.4269274191662777


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Country/config: EG
Dialect: Egyptian Arabic (Cairene) Dialect
Domain: Construction and real estate

English:
Engineer, good morning. Before you run your cables on the third floor, let's coordinate the wall chases.

Reference Arabic:
صباح الخير يا هندسه. قبل ما تمد الكابلات في الدور التالت، خلينا نتفق على مجاري الحيطان.

Prediction:
مهندس، صباح الخير. قبل تبدأين بتلك التوصيلات على الطابق الثالث، دعنا نتنسيق الحفرات في الحائط.


### Generate predictions for all eval set

In [20]:
# ============================================================
# Cell 19 — Generate predictions on FULL eval/test set
# Uses BEST ntkmirror controller loaded in Cell 17
# ============================================================

from pathlib import Path
from tqdm.auto import tqdm
import pandas as pd
import time
import shutil

if "BEST_CONTROLLER_PATH" not in globals():
    raise RuntimeError("Run Cell 17 first. The best ntkmirror controller is not loaded.")

if "generate_translation_from_row" not in globals():
    raise RuntimeError("Run Cell 18 before Cell 19.")

EVAL_LIMIT = None

SAVE_EVERY = 50
STORE_RAW_OUTPUT = False

OVERWRITE_EXISTING = False
BACKUP_BEFORE_OVERWRITE = True

eval_tag = f"best_step{BEST_STEP}"

pred_path = PRED_DIR / f"full_eval_predictions_{EXPERIMENT_NAME}_{eval_tag}.csv"

print("Experiment:", EXPERIMENT_NAME)
print("Using BEST checkpoint:", BEST_CHECKPOINT_PATH)
print("Using BEST controller:", BEST_CONTROLLER_PATH)
print("Best step:", BEST_STEP)
print("Best eval_loss:", BEST_EVAL_LOSS)
print("Saving predictions to:", pred_path)
print("OVERWRITE_EXISTING:", OVERWRITE_EXISTING)

full_eval_df = eval_df.reset_index(drop=True).copy()

if EVAL_LIMIT is not None:
    full_eval_df = full_eval_df.iloc[:EVAL_LIMIT].copy()

expected_n = len(full_eval_df)

print("Total eval/test examples to evaluate:", expected_n)

if pred_path.exists() and OVERWRITE_EXISTING:
    if BACKUP_BEFORE_OVERWRITE:
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        backup_path = pred_path.with_suffix(f".backup_{timestamp}.csv")
        shutil.copy2(pred_path, backup_path)
        print("Existing prediction file backed up to:", backup_path)

    pred_rows = []
    done_ids = set()
    print("FORCING fresh prediction generation from scratch.")

elif pred_path.exists() and not OVERWRITE_EXISTING:
    existing_df = pd.read_csv(pred_path)

    if "source_id" in existing_df.columns:
        existing_df["source_id"] = existing_df["source_id"].astype(str)

        expected_ids = set(full_eval_df["source_id"].astype(str).tolist())
        existing_df = existing_df[existing_df["source_id"].isin(expected_ids)].copy()
        existing_df = existing_df.drop_duplicates(subset=["source_id"], keep="first")

        pred_rows = existing_df.to_dict("records")
        done_ids = set(existing_df["source_id"].astype(str).tolist())

        print(f"Resuming from existing file: {len(done_ids)} examples already done.")

    else:
        print("Existing file has no source_id column. Starting from scratch.")
        pred_rows = []
        done_ids = set()

else:
    pred_rows = []
    done_ids = set()
    print("No existing prediction file. Starting from scratch.")

start_time = time.time()

for _, row in tqdm(full_eval_df.iterrows(), total=len(full_eval_df)):
    row_dict = row.to_dict()
    source_id = str(row_dict["source_id"])

    if source_id in done_ids:
        continue

    try:
        pred, raw = generate_translation_from_row(row_dict)

        out_row = {
            "source_id": row_dict["source_id"],
            "config": row_dict.get("config", ""),
            "dialect": row_dict.get("dialect", ""),
            "domain": row_dict.get("domain", ""),
            "source_text": row_dict["source_text"],
            "reference_arabic": row_dict["target_arabic"],
            "prediction": pred,
            "model_checkpoint": str(BEST_CHECKPOINT_PATH),
            "controller_path": str(BEST_CONTROLLER_PATH),
            "best_step": BEST_STEP,
            "best_eval_loss": BEST_EVAL_LOSS,
            "experiment_name": EXPERIMENT_NAME,
            "adapter_type": "ntkmirror_signed_log_mask_controller",
            "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        }

        if STORE_RAW_OUTPUT:
            out_row["raw_output"] = raw

    except Exception as e:
        out_row = {
            "source_id": row_dict.get("source_id", ""),
            "config": row_dict.get("config", ""),
            "dialect": row_dict.get("dialect", ""),
            "domain": row_dict.get("domain", ""),
            "source_text": row_dict.get("source_text", ""),
            "reference_arabic": row_dict.get("target_arabic", ""),
            "prediction": "",
            "generation_error": repr(e),
            "model_checkpoint": str(BEST_CHECKPOINT_PATH),
            "controller_path": str(BEST_CONTROLLER_PATH),
            "best_step": BEST_STEP,
            "best_eval_loss": BEST_EVAL_LOSS,
            "experiment_name": EXPERIMENT_NAME,
            "adapter_type": "ntkmirror_signed_log_mask_controller",
            "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        }

    pred_rows.append(out_row)
    done_ids.add(source_id)

    if len(pred_rows) % SAVE_EVERY == 0:
        tmp_df = pd.DataFrame(pred_rows)
        tmp_df.to_csv(pred_path, index=False, encoding="utf-8-sig")
        print(f"Saved partial predictions: {len(tmp_df)} rows")

pred_df = pd.DataFrame(pred_rows)

order_df = full_eval_df[["source_id"]].copy()
order_df["source_id"] = order_df["source_id"].astype(str)
order_df["eval_order"] = range(len(order_df))

pred_df["source_id"] = pred_df["source_id"].astype(str)
pred_df = pred_df.merge(order_df, on="source_id", how="left")
pred_df = pred_df.sort_values("eval_order").drop(columns=["eval_order"])
pred_df = pred_df.reset_index(drop=True)

pred_df.to_csv(pred_path, index=False, encoding="utf-8-sig")

elapsed = time.time() - start_time

expected_ids = set(full_eval_df["source_id"].astype(str).tolist())
actual_ids = set(pred_df["source_id"].astype(str).tolist())

missing_ids = expected_ids - actual_ids
extra_ids = actual_ids - expected_ids

print("\nDone.")
print("Saved predictions to:", pred_path)
print("Total rows saved:", len(pred_df))
print("Expected eval/test rows:", expected_n)
print(f"Elapsed time: {elapsed / 60:.2f} minutes")

if missing_ids:
    raise RuntimeError(f"Prediction file is incomplete. Missing {len(missing_ids)} eval examples.")

if extra_ids:
    print(f"Warning: prediction file has {len(extra_ids)} extra source_ids not in current eval_df.")

if len(pred_df) == expected_n:
    print("Full eval/test set was evaluated successfully.")
else:
    print("Warning: row count differs from expected eval size.")

display(pred_df.head())

Experiment: qwen3_4b_alexandria_eg_only_context3_complete2shot_ntkmirror_g10000_score256_mlgate010_lr1e3_all_10epochs
Using BEST checkpoint: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_alexandria_eg_only_context3_complete2shot_ntkmirror_g10000_score256_mlgate010_lr1e3_all_10epochs/checkpoint-1300
Using BEST controller: /content/drive/MyDrive/alexandria_qwen35_sft/final_adapters/qwen3_4b_alexandria_eg_only_context3_complete2shot_ntkmirror_g10000_score256_mlgate010_lr1e3_all_10epochs/qwen3_4b_alexandria_eg_only_context3_complete2shot_ntkmirror_g10000_score256_mlgate010_lr1e3_all_10epochs_best_step1300.pt
Best step: 1300
Best eval_loss: 2.4269274191662777
Saving predictions to: /content/drive/MyDrive/alexandria_qwen35_sft/predictions/full_eval_predictions_qwen3_4b_alexandria_eg_only_context3_complete2shot_ntkmirror_g10000_score256_mlgate010_lr1e3_all_10epochs_best_step1300.csv
OVERWRITE_EXISTING: False
Total eval/test examples to evaluate: 1118
No existing prediction file. 

  0%|          | 0/1118 [00:00<?, ?it/s]

Saved partial predictions: 50 rows
Saved partial predictions: 100 rows
Saved partial predictions: 150 rows
Saved partial predictions: 200 rows
Saved partial predictions: 250 rows
Saved partial predictions: 300 rows
Saved partial predictions: 350 rows
Saved partial predictions: 400 rows
Saved partial predictions: 450 rows
Saved partial predictions: 500 rows
Saved partial predictions: 550 rows
Saved partial predictions: 600 rows
Saved partial predictions: 650 rows
Saved partial predictions: 700 rows
Saved partial predictions: 750 rows
Saved partial predictions: 800 rows
Saved partial predictions: 850 rows
Saved partial predictions: 900 rows
Saved partial predictions: 950 rows
Saved partial predictions: 1000 rows
Saved partial predictions: 1050 rows
Saved partial predictions: 1100 rows

Done.
Saved predictions to: /content/drive/MyDrive/alexandria_qwen35_sft/predictions/full_eval_predictions_qwen3_4b_alexandria_eg_only_context3_complete2shot_ntkmirror_g10000_score256_mlgate010_lr1e3_all_1

,source_id,config,dialect,domain,source_text,reference_arabic,prediction,model_checkpoint,controller_path,best_step,best_eval_loss,experiment_name,adapter_type,generated_at
0,EG_test_EG_test_0_0,EG,Egyptian Arabic (Cairene) Dialect,Commerce and transactions,"I would like one order of kunafa, please.",عايز واحد كنافة لو سمحت.,أنا أريد طبق واحد من كونافا، مين.,/content/drive/MyDrive/alexandria_qwen35_sft/r...,/content/drive/MyDrive/alexandria_qwen35_sft/f...,1300,2.426927,qwen3_4b_alexandria_eg_only_context3_complete2...,ntkmirror_signed_log_mask_controller,2026-06-11 07:22:45
1,EG_test_EG_test_0_1,EG,Egyptian Arabic (Cairene) Dialect,Commerce and transactions,Certainly. Would you like that with cheese or ...,أكيد. تحبها بالجبنة ولا بالقشطة؟,بالطبع. ما شو تفضلين مع الجبن أو مع الزبد؟,/content/drive/MyDrive/alexandria_qwen35_sft/r...,/content/drive/MyDrive/alexandria_qwen35_sft/f...,1300,2.426927,qwen3_4b_alexandria_eg_only_context3_complete2...,ntkmirror_signed_log_mask_controller,2026-06-11 07:22:48
2,EG_test_EG_test_0_2,EG,Egyptian Arabic (Cairene) Dialect,Commerce and transactions,"With cream, please.",بالقشطة، لو سمحت.,مع خلطة الزيت، يرجى.,/content/drive/MyDrive/alexandria_qwen35_sft/r...,/content/drive/MyDrive/alexandria_qwen35_sft/f...,1300,2.426927,qwen3_4b_alexandria_eg_only_context3_complete2...,ntkmirror_signed_log_mask_controller,2026-06-11 07:22:51
3,EG_test_EG_test_1_0,EG,Egyptian Arabic (Cairene) Dialect,Commerce and transactions,"Pardon me, I believe the meat is overcooked. I...",لو سمحت، أعتقد اللحمة مستوية زيادة. ناشفة جدا.,يا فندم، أنا أصدقك، الحيوانات معلقة جدًا، صعبة...,/content/drive/MyDrive/alexandria_qwen35_sft/r...,/content/drive/MyDrive/alexandria_qwen35_sft/f...,1300,2.426927,qwen3_4b_alexandria_eg_only_context3_complete2...,ntkmirror_signed_log_mask_controller,2026-06-11 07:22:54
4,EG_test_EG_test_1_1,EG,Egyptian Arabic (Cairene) Dialect,Commerce and transactions,"I'm very sorry to hear that, sir. Would you li...",آسفه جدا يا فندم. تحب أخلي الشيف يجهزلك واحدة ...,أنا مش عارف ما حد يقدر يساعدني في التسوق، أنا ...,/content/drive/MyDrive/alexandria_qwen35_sft/r...,/content/drive/MyDrive/alexandria_qwen35_sft/f...,1300,2.426927,qwen3_4b_alexandria_eg_only_context3_complete2...,ntkmirror_signed_log_mask_controller,2026-06-11 07:22:58


### **Compute BLEU and chrF**

In [21]:
# ============================================================
# Cell 20 — Compute BLEU, chrF, chrF++, and spBLEU
# Robust SacreBLEU version: no crash if .signature is unavailable
# ============================================================

import pandas as pd
import json
from pathlib import Path

try:
    import sacrebleu
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sacrebleu", "sentencepiece"])
    import sacrebleu

from sacrebleu.metrics import BLEU, CHRF

if "BEST_STEP" not in globals():
    raise RuntimeError("BEST_STEP is not defined. Run Cell 17 first.")

eval_tag = f"best_step{BEST_STEP}"

pred_path = PRED_DIR / f"full_eval_predictions_{EXPERIMENT_NAME}_{eval_tag}.csv"
metrics_path = PRED_DIR / f"full_eval_metrics_{EXPERIMENT_NAME}_{eval_tag}.json"

print("Experiment:", EXPERIMENT_NAME)
print("Best checkpoint:", BEST_CHECKPOINT_PATH)
print("Best controller:", BEST_CONTROLLER_PATH)
print("Best step:", BEST_STEP)
print("Prediction file:", pred_path)
print("Metrics file:", metrics_path)

if not pred_path.exists():
    raise FileNotFoundError(f"Prediction file not found: {pred_path}")

pred_df = pd.read_csv(pred_path)

required_cols = {"source_id", "prediction", "reference_arabic"}
missing = required_cols - set(pred_df.columns)

if missing:
    raise ValueError(f"Missing required columns in prediction file: {missing}")

pred_df["source_id"] = pred_df["source_id"].astype(str)
pred_df["prediction"] = pred_df["prediction"].fillna("").astype(str)
pred_df["reference_arabic"] = pred_df["reference_arabic"].fillna("").astype(str)

expected_eval_df = eval_df.reset_index(drop=True).copy()
expected_eval_df["source_id"] = expected_eval_df["source_id"].astype(str)

expected_ids = set(expected_eval_df["source_id"].tolist())
actual_ids = set(pred_df["source_id"].tolist())

missing_ids = expected_ids - actual_ids
extra_ids = actual_ids - expected_ids

print("\n==============================")
print("Full Eval/Test Coverage Check")
print("==============================")
print("Expected eval/test examples:", len(expected_eval_df))
print("Prediction rows:", len(pred_df))
print("Unique prediction source_ids:", len(actual_ids))

if missing_ids:
    raise RuntimeError(
        f"Prediction file is NOT full eval/test. "
        f"Missing {len(missing_ids)} examples. "
        f"Run Cell 19 again to finish generation."
    )

if extra_ids:
    print(f"Warning: prediction file has {len(extra_ids)} extra source_ids not in current eval_df.")

print("Full eval/test coverage confirmed.")

order_df = expected_eval_df[["source_id"]].copy()
pred_df_ordered = order_df.merge(pred_df, on="source_id", how="left")

preds = pred_df_ordered["prediction"].fillna("").astype(str).tolist()
refs = pred_df_ordered["reference_arabic"].fillna("").astype(str).tolist()

def safe_signature(metric):
    try:
        return str(metric.get_signature())
    except Exception:
        return None

def compute_lexical_metrics(preds, refs):
    """
    Computes:
    - BLEU: standard sacrebleu BLEU
    - chrF: character F-score
    - chrF++: chrF with word_order=2
    - spBLEU: BLEU with FLORES200/SentencePiece tokenizer when available
    """

    bleu_metric = BLEU()
    chrf_metric = CHRF(word_order=0)
    chrfpp_metric = CHRF(word_order=2)

    bleu = bleu_metric.corpus_score(preds, [refs])
    chrf = chrf_metric.corpus_score(preds, [refs])
    chrfpp = chrfpp_metric.corpus_score(preds, [refs])

    spbleu_score = None
    spbleu_signature = None

    try:
        spbleu_metric = BLEU(tokenize="flores200")
        spbleu = spbleu_metric.corpus_score(preds, [refs])
        spbleu_score = float(spbleu.score)
        spbleu_signature = safe_signature(spbleu_metric)
    except Exception as e:
        print("WARNING: spBLEU failed:", repr(e))

    return {
        "BLEU": float(bleu.score),
        "chrF": float(chrf.score),
        "chrF++": float(chrfpp.score),
        "spBLEU": spbleu_score,
        "BLEU_signature": safe_signature(bleu_metric),
        "chrF_signature": safe_signature(chrf_metric),
        "chrF++_signature": safe_signature(chrfpp_metric),
        "spBLEU_signature": spbleu_signature,
    }

overall_metrics = compute_lexical_metrics(preds, refs)

metrics = {
    "experiment": EXPERIMENT_NAME,
    "adapter_type": "ntkmirror_signed_log_mask_controller",
    "checkpoint": str(BEST_CHECKPOINT_PATH),
    "controller": str(BEST_CONTROLLER_PATH),
    "best_step": int(BEST_STEP),
    "best_eval_loss": float(BEST_EVAL_LOSS),
    "num_examples": len(pred_df_ordered),
    "prediction_file": str(pred_path),
    **overall_metrics,
}

print("\n==============================")
print("Full Eval/Test Metrics from BEST ntkmirror controller")
print("==============================")
print(f"Examples: {len(pred_df_ordered)}")
print(f"BLEU:     {metrics['BLEU']:.4f}")
print(f"chrF:     {metrics['chrF']:.4f}")
print(f"chrF++:   {metrics['chrF++']:.4f}")

if metrics["spBLEU"] is not None:
    print(f"spBLEU:   {metrics['spBLEU']:.4f}")
else:
    print("spBLEU:   FAILED / unavailable")

def compute_group_metrics(df, group_col):
    rows = []

    if group_col not in df.columns:
        return pd.DataFrame(rows)

    for group_value in sorted(df[group_col].dropna().unique()):
        tmp = df[df[group_col] == group_value]

        if len(tmp) == 0:
            continue

        group_preds = tmp["prediction"].fillna("").astype(str).tolist()
        group_refs = tmp["reference_arabic"].fillna("").astype(str).tolist()

        row = {
            group_col: group_value,
            "num_examples": len(tmp),
        }

        row.update(compute_lexical_metrics(group_preds, group_refs))
        rows.append(row)

    return pd.DataFrame(rows)

per_config_df = compute_group_metrics(pred_df_ordered, "config")
per_dialect_df = compute_group_metrics(pred_df_ordered, "dialect")
per_domain_df = compute_group_metrics(pred_df_ordered, "domain")

if len(per_config_df):
    print("\n==============================")
    print("Per-config Metrics")
    print("==============================")
    display(per_config_df)
    metrics["per_config"] = per_config_df.to_dict("records")

if len(per_dialect_df):
    print("\n==============================")
    print("Per-dialect Metrics")
    print("==============================")
    display(per_dialect_df)
    metrics["per_dialect"] = per_dialect_df.to_dict("records")

if len(per_domain_df):
    print("\n==============================")
    print("Per-domain Metrics")
    print("==============================")
    display(per_domain_df)
    metrics["per_domain"] = per_domain_df.to_dict("records")

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print("\nSaved metrics to:")
print(metrics_path)

display(pred_df_ordered[["source_text", "reference_arabic", "prediction"]].head(10))

Experiment: qwen3_4b_alexandria_eg_only_context3_complete2shot_ntkmirror_g10000_score256_mlgate010_lr1e3_all_10epochs
Best checkpoint: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_alexandria_eg_only_context3_complete2shot_ntkmirror_g10000_score256_mlgate010_lr1e3_all_10epochs/checkpoint-1300
Best controller: /content/drive/MyDrive/alexandria_qwen35_sft/final_adapters/qwen3_4b_alexandria_eg_only_context3_complete2shot_ntkmirror_g10000_score256_mlgate010_lr1e3_all_10epochs/qwen3_4b_alexandria_eg_only_context3_complete2shot_ntkmirror_g10000_score256_mlgate010_lr1e3_all_10epochs_best_step1300.pt
Best step: 1300
Prediction file: /content/drive/MyDrive/alexandria_qwen35_sft/predictions/full_eval_predictions_qwen3_4b_alexandria_eg_only_context3_complete2shot_ntkmirror_g10000_score256_mlgate010_lr1e3_all_10epochs_best_step1300.csv
Metrics file: /content/drive/MyDrive/alexandria_qwen35_sft/predictions/full_eval_metrics_qwen3_4b_alexandria_eg_only_context3_complete2shot_ntkmirror_g

,config,num_examples,BLEU,chrF,chrF++,spBLEU,BLEU_signature,chrF_signature,chrF++_signature,spBLEU_signature
0,EG,1118,2.926224,26.591606,23.551955,8.230084,nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|v...,nrefs:1|case:mixed|eff:yes|nc:6|nw:0|space:no|...,nrefs:1|case:mixed|eff:yes|nc:6|nw:2|space:no|...,nrefs:1|case:mixed|eff:no|tok:flores200|smooth...



Per-dialect Metrics


,dialect,num_examples,BLEU,chrF,chrF++,spBLEU,BLEU_signature,chrF_signature,chrF++_signature,spBLEU_signature
0,Egyptian Arabic (Cairene) Dialect,1118,2.926224,26.591606,23.551955,8.230084,nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|v...,nrefs:1|case:mixed|eff:yes|nc:6|nw:0|space:no|...,nrefs:1|case:mixed|eff:yes|nc:6|nw:2|space:no|...,nrefs:1|case:mixed|eff:no|tok:flores200|smooth...



Per-domain Metrics


,domain,num_examples,BLEU,chrF,chrF++,spBLEU,BLEU_signature,chrF_signature,chrF++_signature,spBLEU_signature
0,Agriculture and farming,100,2.199476,24.847433,21.944247,7.192936,nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|v...,nrefs:1|case:mixed|eff:yes|nc:6|nw:0|space:no|...,nrefs:1|case:mixed|eff:yes|nc:6|nw:2|space:no|...,nrefs:1|case:mixed|eff:no|tok:flores200|smooth...
1,Commerce and transactions,103,1.790136,22.776032,20.357081,6.158699,nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|v...,nrefs:1|case:mixed|eff:yes|nc:6|nw:0|space:no|...,nrefs:1|case:mixed|eff:yes|nc:6|nw:2|space:no|...,nrefs:1|case:mixed|eff:no|tok:flores200|smooth...
2,Construction and real estate,102,2.309566,27.936982,24.416824,8.192348,nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|v...,nrefs:1|case:mixed|eff:yes|nc:6|nw:0|space:no|...,nrefs:1|case:mixed|eff:yes|nc:6|nw:2|space:no|...,nrefs:1|case:mixed|eff:no|tok:flores200|smooth...
3,Education and academia,102,4.135515,26.964633,24.221197,8.624200,nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|v...,nrefs:1|case:mixed|eff:yes|nc:6|nw:0|space:no|...,nrefs:1|case:mixed|eff:yes|nc:6|nw:2|space:no|...,nrefs:1|case:mixed|eff:no|tok:flores200|smooth...
4,Energy and resources,103,3.707531,29.707073,26.484782,10.659260,nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|v...,nrefs:1|case:mixed|eff:yes|nc:6|nw:0|space:no|...,nrefs:1|case:mixed|eff:yes|nc:6|nw:2|space:no|...,nrefs:1|case:mixed|eff:no|tok:flores200|smooth...
5,Everyday and social,103,1.915815,21.754291,19.455288,5.050218,nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|v...,nrefs:1|case:mixed|eff:yes|nc:6|nw:0|space:no|...,nrefs:1|case:mixed|eff:yes|nc:6|nw:2|space:no|...,nrefs:1|case:mixed|eff:no|tok:flores200|smooth...
6,Healthcare and medical,102,2.325720,25.453442,22.461350,6.791046,nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|v...,nrefs:1|case:mixed|eff:yes|nc:6|nw:0|space:no|...,nrefs:1|case:mixed|eff:yes|nc:6|nw:2|space:no|...,nrefs:1|case:mixed|eff:no|tok:flores200|smooth...
7,Legal and financial,103,2.809103,29.071292,25.468734,9.413361,nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|v...,nrefs:1|case:mixed|eff:yes|nc:6|nw:0|space:no|...,nrefs:1|case:mixed|eff:yes|nc:6|nw:2|space:no|...,nrefs:1|case:mixed|eff:no|tok:flores200|smooth...
8,Logistics and transportation,100,2.089446,27.694671,24.545669,8.659859,nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|v...,nrefs:1|case:mixed|eff:yes|nc:6|nw:0|space:no|...,nrefs:1|case:mixed|eff:yes|nc:6|nw:2|space:no|...,nrefs:1|case:mixed|eff:no|tok:flores200|smooth...
9,Professional and workplace,100,3.538298,28.391376,25.221298,9.633170,nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|v...,nrefs:1|case:mixed|eff:yes|nc:6|nw:0|space:no|...,nrefs:1|case:mixed|eff:yes|nc:6|nw:2|space:no|...,nrefs:1|case:mixed|eff:no|tok:flores200|smooth...



Saved metrics to:
/content/drive/MyDrive/alexandria_qwen35_sft/predictions/full_eval_metrics_qwen3_4b_alexandria_eg_only_context3_complete2shot_ntkmirror_g10000_score256_mlgate010_lr1e3_all_10epochs_best_step1300.json


,source_text,reference_arabic,prediction
0,"I would like one order of kunafa, please.",عايز واحد كنافة لو سمحت.,أنا أريد طبق واحد من كونافا، مين.
1,Certainly. Would you like that with cheese or ...,أكيد. تحبها بالجبنة ولا بالقشطة؟,بالطبع. ما شو تفضلين مع الجبن أو مع الزبد؟
2,"With cream, please.",بالقشطة، لو سمحت.,مع خلطة الزيت، يرجى.
3,"Pardon me, I believe the meat is overcooked. I...",لو سمحت، أعتقد اللحمة مستوية زيادة. ناشفة جدا.,يا فندم، أنا أصدقك، الحيوانات معلقة جدًا، صعبة...
4,"I'm very sorry to hear that, sir. Would you li...",آسفه جدا يا فندم. تحب أخلي الشيف يجهزلك واحدة ...,أنا مش عارف ما حد يقدر يساعدني في التسوق، أنا ...
5,"Yes, please. Thank you.",أيوه لو سمحتي. شكرا.,نعم، يرجى. شكرا.
6,"Honestly, the driver was rude and the car was ...",بصراحة، السواق كان بجح و العربيه ماكانتش نضيفه...,الحقيقة، الميكروباص كان مزعج والسيارة كانت غير...
7,I truly apologize that we failed to provide yo...,أنا حقيقي بعتذر ان احنا فشلنا اننا نقدملك خدمة...,ياوووووووووووووووووووووووووووووووووووووووووووو...
8,"Engineer, good morning. Before you run your ca...",صباح الخير يا هندسه. قبل ما تمد الكابلات في ال...,مهندس، صباح الخير. قبل تبدأين بتلك التوصيلات ع...
9,"Good thing you caught me, I was about to start...",كويس إنك لحقتني، أنا كنت هابدأ. هتحط خطوط المي...,حسنك انتا بعتني، كنت عندي حاجة. أين بتوضع خطوط...


### **Error-analysis view by country/domain**

In [22]:
# ============================================================
# Cell 21 — Qualitative error-analysis samples
# Uses FULL eval predictions from BEST checkpoint
# ============================================================

if "BEST_STEP" not in globals():
    raise RuntimeError("BEST_STEP is not defined. Run Cell 17 first.")

eval_tag = f"best_step{BEST_STEP}"

analysis_path = PRED_DIR / f"manual_analysis_{EXPERIMENT_NAME}_{eval_tag}.xlsx"

if "pred_df_ordered" in globals():
    analysis_df = pred_df_ordered.copy()
elif "pred_df" in globals():
    analysis_df = pred_df.copy()
else:
    raise RuntimeError("No prediction dataframe found. Run Cell 19/20 first.")

with pd.ExcelWriter(analysis_path, engine="openpyxl") as writer:
    analysis_df.to_excel(writer, sheet_name="all_predictions", index=False)

    if "config" in analysis_df.columns:
        for cfg in analysis_df["config"].dropna().unique()[:10]:
            tmp = analysis_df[analysis_df["config"] == cfg]
            tmp.to_excel(writer, sheet_name=str(cfg)[:31], index=False)

print("Saved manual analysis workbook to:")
print(analysis_path)

Saved manual analysis workbook to:
/content/drive/MyDrive/alexandria_qwen35_sft/predictions/manual_analysis_qwen3_4b_alexandria_eg_only_context3_complete2shot_ntkmirror_g10000_score256_mlgate010_lr1e3_all_10epochs_best_step1300.xlsx
